# Model Building

# Silver

### Helper Functions

In [0]:
from pyspark.sql import functions as F
import re
def flatten_two_row_header(path):
    df_raw = spark.read.option("header", "false").csv(path)
    header_rows = df_raw.limit(2).collect()
    top_header, sub_header = header_rows[0], header_rows[1]
    n_cols = len(df_raw.columns)

    filled_top, last = [], ""
    for i in range(n_cols):
        val = top_header[i]
        if val is not None and val.strip() != "":
            last = val.strip()
        filled_top.append(last)

    new_columns = []
    for i in range(n_cols):
        top, sub = filled_top[i], (sub_header[i].strip() if sub_header[i] else "")
        new_columns.append(f"{top}_{sub}" if top and sub else (sub or top or f"col_{i}"))

    return (
        df_raw
        .withColumn("_row_num", F.monotonically_increasing_id())
        .filter(F.col("_row_num") >= 2)
        .drop("_row_num")
        .toDF(*new_columns)
    )

def sanitize_columns(df):
    for old_name in df.columns:
        new_name = re.sub(r'[ ,;{}()\n\t=]+', '_', old_name)
        if new_name != old_name:
            df = df.withColumnRenamed(old_name, new_name)
    return df

def path_exists(path):
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False
    
PROJECT_START_SEASON_YEAR=2025

## delta table update 

In [0]:
%sql
-- ALTER TABLE delta.`s3://football-data-raw-01/gold/fact_match_summary` ADD COLUMN (season_name STRING)

## dim_season

In [0]:
from pyspark.sql import functions as F, Window
from delta.tables import DeltaTable
from datetime import date

dim_season_path = "s3://football-data-raw-01/silver/dim_season"
SEASON_ID_BASE = 100000

football_data_org_seasons_path = "s3://football-data-raw-01/raw/2025/football_data_org/leagues.json"
sports_api_pro_seasons_path = "s3://football-data-raw-01/raw/2025/sports_api_pro/seasons.json"

soccerdata_fbref_seasons_paths = [
    f"s3://football-data-raw-01/raw/{year}/soccerdata_fbref/schedule.csv"
    for year in range(PROJECT_START_SEASON_YEAR, date.today().year + 1)
]
soccerdata_fbref_seasons_paths = [p for p in soccerdata_fbref_seasons_paths if path_exists(p)]

df_football_data_org_seasons = spark.read.option("multiLine", "true").json(football_data_org_seasons_path)
df_sports_api_pro_seasons = spark.read.option("multiLine", "true").json(sports_api_pro_seasons_path)
df_soccerdata_fbref_seasons = spark.read.option("inferSchema", "true").option("header", "true").csv(soccerdata_fbref_seasons_paths)

df_football_data_org_seasons = (
    df_football_data_org_seasons
    .select(F.explode("seasons").alias("season"))
    .select(
        F.col("season.id").alias("football_data_org_season_id"),
        F.to_date(F.col("season.startDate")).alias("start_date"),
        F.to_date(F.col("season.endDate")).alias("end_date")
    )
    .withColumn(
        "season_name",
        F.concat(
            F.year(F.col("start_date")).cast("string"),
            F.lit("/"),
            F.date_format(F.col("end_date"), "yy")
        )
    )
    .filter(F.year(F.col("start_date")) >= PROJECT_START_SEASON_YEAR)
    .orderBy(F.col("start_date").desc())
)

df_sports_api_pro_seasons = (
    df_sports_api_pro_seasons
    .select(F.explode("seasons").alias("season"))
    .select(
        F.col("season.id").alias("sports_api_pro_season_id"),
        F.col("season.tournamentId").alias("tournamentId"),
        F.col("season.year").alias("year")
    )
    .withColumn(
        "season_name",
        F.concat(
            F.lit("20"),
            F.col("year").cast("string")
        )
    )
    .filter(F.substring(F.col("year"), 1, 2).cast("int") + 2000 >= PROJECT_START_SEASON_YEAR)
    .drop("year")
)

df_soccerdata_fbref_seasons = (
    df_soccerdata_fbref_seasons
    .select(F.col("season").alias("fbref_season_id"), F.col("date"))
    .groupBy("fbref_season_id").agg(F.min("date").alias("start_date"), F.max("date").alias("end_date"))
    .withColumn(
        "season_name"
        ,F.concat(
            F.year(F.col("start_date")).cast("string"),
            F.lit("/"),
            F.date_format(F.col("end_date"), "yy")
        )
    )
    .orderBy(F.col("start_date").desc())
    .drop("end_date","start_date")   
)

dim_season = (
    df_football_data_org_seasons.join(df_soccerdata_fbref_seasons, on="season_name", how="inner")
    .join(df_sports_api_pro_seasons, on="season_name", how="inner")
)


if DeltaTable.isDeltaTable(spark, dim_season_path):
    existing = spark.read.format("delta").load(dim_season_path).select("season_name", "season_id")
    max_id = existing.agg(F.max("season_id")).collect()[0][0] or SEASON_ID_BASE

    new_seasons = dim_season.join(existing, on="season_name", how="left_anti")

    if new_seasons.count() > 0:
        new_seasons = new_seasons.withColumn(
            "season_id",
            max_id + F.row_number().over(Window.orderBy("season_name"))
        )
        delta_table = DeltaTable.forPath(spark, dim_season_path)
        (
            delta_table.alias("t")
            .merge(new_seasons.alias("s"), "t.season_name = s.season_name")
            .whenNotMatchedInsertAll()
            .execute()
        )
    else:
        dim_season = spark.read.format("delta").load(dim_season_path)
else:
    dim_season = dim_season.withColumn(
        "season_id",
        SEASON_ID_BASE + F.row_number().over(Window.orderBy("season_name"))
    )
    dim_season.write.format("delta").mode("overwrite").save(dim_season_path)

display(spark.read.format("delta").load(dim_season_path))

## dim_team

In [0]:
from pyspark.sql import functions as F, Window
from delta.tables import DeltaTable
from datetime import date

TEAM_ID_BASE = 200000
PROJECT_START_SEASON_YEAR = 2025
dim_team_path = "s3://football-data-raw-01/silver/dim_team"

football_data_org_teams_paths = [
    f"s3://football-data-raw-01/raw/{year}/football_data_org/teams.json"
    for year in range(PROJECT_START_SEASON_YEAR, date.today().year + 1)
]
football_data_org_teams_paths = [p for p in football_data_org_teams_paths if path_exists(p)]

sports_api_pro_teams_paths = [
    f"s3://football-data-raw-01/raw/{year}/sports_api_pro/standings.json"
    for year in range(PROJECT_START_SEASON_YEAR, date.today().year + 1)
]
sports_api_pro_teams_paths = [p for p in sports_api_pro_teams_paths if path_exists(p)]

soccerdata_fbref_teams_paths = [
    f"s3://football-data-raw-01/raw/{year}/soccerdata_fbref/team_season_standard_stats.csv"
    for year in range(PROJECT_START_SEASON_YEAR, date.today().year + 1)
]
soccerdata_fbref_teams_paths = [p for p in soccerdata_fbref_teams_paths if path_exists(p)]

df_football_data_org_teams = spark.read.option("multiLine", "true").json(football_data_org_teams_paths)
df_sports_api_pro_teams = spark.read.option("multiLine", "true").json(sports_api_pro_teams_paths)
df_soccerdata_fbref_teams = spark.read.option("inferSchema", "true").option("header", "true").csv(soccerdata_fbref_teams_paths)

df_football_data_org_teams = (
    df_football_data_org_teams
    .select(F.col("season"),F.explode(F.col("teams")).alias("teams"))
    .select(F.col("season.id").alias("football_data_org_season_id"),
            F.col("teams.id").alias("football_data_org_team_id"),
            F.col("teams.name").alias("team_name"),
            F.col("teams.shortName").alias("team_short_name"),
            F.col("teams.tla").alias("team_tla"),
            F.col("teams.crest").alias("team_crest"),
            F.col("teams.venue").alias("team_venue"),
            )
    .withColumn(
        "rn",
        F.row_number().over(
            Window.partitionBy("football_data_org_team_id").orderBy(F.col("football_data_org_season_id").desc())
        )
    )
    .filter(F.col("rn") == 1)
    .drop("rn")
)

df_sports_api_pro_teams = (
    df_sports_api_pro_teams
    .select(F.col("seasonId").alias("sports_api_pro_season_id"),F.explode(F.col("standings")).alias("standings"))
    .select(F.col("sports_api_pro_season_id"),
            F.col("standings.teamId").alias("sports_api_pro_team_id"),
            F.col("standings.teamName").alias("team_name"))
    .withColumn(
        "rn",
        F.row_number().over(
            Window.partitionBy("sports_api_pro_team_id").orderBy(F.col("sports_api_pro_season_id").desc())
        )
    )
    .filter(F.col("rn") == 1)
    .drop("rn")
    )

team_tla_mapping = (
    df_sports_api_pro_teams.select("team_name").distinct().alias("sp")
    .crossJoin(
        df_football_data_org_teams.select("team_name", "team_tla").distinct().alias("fd")
    )
    .filter(F.col("fd.team_name").contains(F.col("sp.team_name")))
    .select(
        F.col("sp.team_name").alias("team_name"),
        F.col("fd.team_tla").alias("team_tla")
    )
)

df_sports_api_pro_teams = (
    df_sports_api_pro_teams
    .join(team_tla_mapping, on="team_name", how="left")
)

dim_team = (
    df_football_data_org_teams.alias("fd")
    .join(df_sports_api_pro_teams.alias("sp"), on="team_tla", how="left")
    .select(
        F.col("football_data_org_team_id"),
        F.col("team_short_name"),
        F.col("team_tla"),
        F.col("fd.team_name").alias("team_name"),
        F.col("team_crest"),
        F.col("team_venue"),
        F.col("sports_api_pro_team_id")
    )
)

df_soccerdata_fbref_teams = df_soccerdata_fbref_teams.dropna()
df_soccerdata_fbref_teams = (
    df_soccerdata_fbref_teams
    .select(F.col("season").alias("fbref_season_id"),
            F.col("team").alias("team_name"),
            F.col("url"))
    .withColumn(
    "fbref_team_id", F.regexp_extract(F.col("url"), r"/squads/([a-f0-9]+)/", 1)
    ).withColumn(
    "name", F.regexp_extract(F.col("url"), r"/([^/]+)$", 1)
    )
    .withColumn(
    "name", F.regexp_replace(F.col("name"), "-Stats$", "")
    )
    .withColumn(
    "name", F.regexp_replace(F.col("name"), "-", " ")
    )
    .withColumn(
        "rn",
        F.row_number().over(
            Window.partitionBy("fbref_team_id").orderBy(F.col("fbref_season_id").desc())
        )
    )
    .filter(F.col("rn") == 1)
    .drop("rn")
)

dim_team = (
    dim_team.alias("dt")
    .join(
        df_soccerdata_fbref_teams.alias("fb"),
        (
            F.col("dt.team_name").contains(F.col("fb.team_name")) |
            F.col("fb.team_name").contains(F.col("dt.team_name")) |
            F.col("dt.team_name").contains(F.col("fb.name")) |
            F.col("fb.name").contains(F.col("dt.team_name")) |
            F.col("dt.team_short_name").contains(F.col("fb.team_name")) |
            F.col("fb.team_name").contains(F.col("dt.team_short_name")) |
            F.col("dt.team_short_name").contains(F.col("fb.name")) |
            F.col("fb.name").contains(F.col("dt.team_short_name"))
        ),
        "left"
    )
    .select(
        "dt.*",
        F.col("fb.fbref_team_id"),
        F.col("fb.team_name").alias("fbref_team_name")
    )
)

if DeltaTable.isDeltaTable(spark, dim_team_path):
    existing = spark.read.format("delta").load(dim_team_path).select("team_tla", "team_id")
    max_id = existing.agg(F.max("team_id")).collect()[0][0] or TEAM_ID_BASE

    new_teams = dim_team.join(existing, on="team_tla", how="left_anti")

    if new_teams.count() > 0:
        new_teams = new_teams.withColumn(
            "team_id",
            max_id + F.row_number().over(Window.orderBy("team_tla"))
        )
        delta_table = DeltaTable.forPath(spark, dim_team_path)
        (
            delta_table.alias("t")
            .merge(new_teams.alias("s"), "t.team_tla = s.team_tla")
            .whenNotMatchedInsertAll()
            .execute()
        )

    dim_team_with_id = dim_team.join(existing, on="team_tla", how="inner")

    delta_table = DeltaTable.forPath(spark, dim_team_path)
    (
        delta_table.alias("t")
        .merge(dim_team_with_id.alias("s"), "t.team_tla = s.team_tla")
        .whenMatchedUpdateAll()
        .execute()
    )

    dim_team = spark.read.format("delta").load(dim_team_path)
else:
    dim_team = dim_team.withColumn(
        "team_id",
        TEAM_ID_BASE + F.row_number().over(Window.orderBy("team_tla"))
    )
    dim_team.write.format("delta").mode("overwrite").save(dim_team_path)

display(spark.read.format("delta").load(dim_team_path))

## fact_team_season

In [0]:
from pyspark.sql import functions as F, Window
from delta.tables import DeltaTable

FACT_TEAM_SEASON_ID_BASE = 100500

fact_team_season_path = "s3://football-data-raw-01/silver/fact_team_season"
dim_team_path = "s3://football-data-raw-01/silver/dim_team"
dim_season_path = "s3://football-data-raw-01/silver/dim_season"

dim_team = spark.read.format("delta").load(dim_team_path)
dim_season = spark.read.format("delta").load(dim_season_path)

fact_team_season_new = (
    dim_team.alias("dt")
    .join(
        df_football_data_org_teams.alias("fd"),
        on=["football_data_org_team_id"],
        how="inner"
    )
    .join(
        dim_season.alias("ds"),
        on=["football_data_org_season_id"],
        how="inner"
    )
    .select(
        F.col("ds.season_id"),
        F.col("dt.team_id")
    )
)

if DeltaTable.isDeltaTable(spark, fact_team_season_path):
    existing = spark.read.format("delta").load(fact_team_season_path)
    max_id = existing.agg(F.max("fact_team_season_id")).collect()[0][0] or FACT_TEAM_SEASON_ID_BASE

    new_rows = fact_team_season_new.join(
        existing.select("season_id", "team_id"),
        on=["season_id", "team_id"],
        how="left_anti"
    )

    if new_rows.count() > 0:
        new_rows = new_rows.withColumn(
            "fact_team_season_id",
            max_id + F.row_number().over(Window.orderBy("season_id", "team_id"))
        )
        delta_table = DeltaTable.forPath(spark, fact_team_season_path)
        (
            delta_table.alias("t")
            .merge(new_rows.alias("s"), "t.season_id = s.season_id AND t.team_id = s.team_id")
            .whenNotMatchedInsertAll()
            .execute()
        )
        fact_team_season = spark.read.format("delta").load(fact_team_season_path)
    else:
        fact_team_season = existing
else:
    fact_team_season = fact_team_season_new.withColumn(
        "fact_team_season_id",
        FACT_TEAM_SEASON_ID_BASE + F.row_number().over(Window.orderBy("season_id", "team_id"))
    )
    fact_team_season.write.format("delta").mode("overwrite").save(fact_team_season_path)

display(fact_team_season)


## dim_player

In [0]:
from pyspark.sql import functions as F, Window
from delta.tables import DeltaTable
from datetime import date

PLAYER_ID_BASE = 300000
dim_players_path = "s3://football-data-raw-01/silver/dim_players"


def fbref_paths(file_name):
    paths = [
        f"s3://football-data-raw-01/raw/{year}/soccerdata_fbref/{file_name}"
        for year in range(PROJECT_START_SEASON_YEAR, date.today().year + 1)
    ]
    return [p for p in paths if path_exists(p)]


# Who exists comes from the matchday lineups: they list every squad member,
# including unused subs and players who debuted after the last season-stats
# refresh. Season stats only list players who have appeared, so they just fill
# in nationality/position/birth year. Names are unique across seasons, and the
# lineup/event cells match players by name.
lineup_players = (
    spark.read.option("inferSchema", "true").option("header", "true").csv(fbref_paths("lineup.csv"))
    .select(F.col("player").alias("player_name"))
    .dropna()
    .distinct()
)

season_players = (
    spark.read.option("inferSchema", "true").option("header", "true").csv(fbref_paths("players_season_standard_stats.csv"))
    .select(
        F.col("player").alias("player_name"),
        F.col("nation").alias("player_nationality"),
        F.col("pos").alias("player_position"),
        F.col("born").alias("player_born"),
    )
    .dropna(subset=["player_name"])       # also drops the CSV's second header row
    .dropDuplicates(["player_name"])
)

dim_players_new = lineup_players.join(season_players, on="player_name", how="full_outer")

if DeltaTable.isDeltaTable(spark, dim_players_path):
    existing = spark.read.format("delta").load(dim_players_path)
    max_id = existing.agg(F.max("player_id")).collect()[0][0] or PLAYER_ID_BASE
    delta_table = DeltaTable.forPath(spark, dim_players_path)

    # A player first seen in a lineup gets his details once season stats list him
    # (the season-stats cells join dim_players on name + birth year).
    (
        delta_table.alias("t")
        .merge(dim_players_new.alias("s"), "t.player_name = s.player_name")
        .whenMatchedUpdate(
            condition="t.player_born IS NULL AND s.player_born IS NOT NULL",
            set={
                "player_nationality": "s.player_nationality",
                "player_position": "s.player_position",
                "player_born": "s.player_born",
            },
        )
        .execute()
    )

    new_players = dim_players_new.join(existing.select("player_name"), on="player_name", how="left_anti")
    if new_players.count() > 0:
        new_players = new_players.withColumn(
            "player_id",
            max_id + F.row_number().over(Window.orderBy("player_name"))
        )
        (
            delta_table.alias("t")
            .merge(new_players.alias("s"), "t.player_name = s.player_name")
            .whenNotMatchedInsertAll()
            .execute()
        )
    dim_players = spark.read.format("delta").load(dim_players_path)
else:
    dim_players = dim_players_new.withColumn(
        "player_id",
        PLAYER_ID_BASE + F.row_number().over(Window.orderBy("player_name"))
    )
    dim_players.write.format("delta").mode("overwrite").save(dim_players_path)

display(dim_players)

## fact_player_season

In [0]:
from pyspark.sql import functions as F, Window
from delta.tables import DeltaTable
from datetime import date

FACT_PLAYER_SEASON_ID_BASE = 200500

fact_player_season_path = "s3://football-data-raw-01/silver/fact_player_season"
dim_players_path = "s3://football-data-raw-01/silver/dim_players"
dim_team_path = "s3://football-data-raw-01/silver/dim_team"
dim_season_path = "s3://football-data-raw-01/silver/dim_season"

soccerdata_fbref_players_paths = [
    f"s3://football-data-raw-01/raw/{year}/soccerdata_fbref/players_season_standard_stats.csv"
    for year in range(PROJECT_START_SEASON_YEAR, date.today().year + 1)
]
soccerdata_fbref_players_paths = [p for p in soccerdata_fbref_players_paths if path_exists(p)]

df_soccerdata_fbref_players = spark.read.option("inferSchema", "true").option("header", "true").csv(soccerdata_fbref_players_paths)
dim_players = spark.read.format("delta").load(dim_players_path)
dim_team = spark.read.format("delta").load(dim_team_path)
dim_season = spark.read.format("delta").load(dim_season_path)

df_soccerdata_fbref_players = (
    df_soccerdata_fbref_players
    .dropna()
    .withColumnRenamed("season", "fbref_season_id")
    .select(
        F.col("player"),
        F.col("fbref_season_id"),
        F.col("team"),
        F.col("born")
    )
)

fact_player_season_new = (
    df_soccerdata_fbref_players.alias("ds")
    .join(dim_players.alias("dp"),
        (F.col("ds.player") == F.col("dp.player_name")) & (F.col("ds.born") == F.col("dp.player_born")),
        how="inner")
    .join(
        dim_team.alias("dt"),
        F.col("ds.team") == F.col("dt.fbref_team_name"),
        "inner"
        )
    .join(
        dim_season.alias("dsn"),
        on=["fbref_season_id"],
        how="inner"
    )
    .select(
        F.col("dsn.season_id"),
        F.col("dp.player_id"),
        F.col("dt.team_id"),
        )
    )

if DeltaTable.isDeltaTable(spark, fact_player_season_path):
    existing = spark.read.format("delta").load(fact_player_season_path)
    max_id = existing.agg(F.max("fact_player_season_id")).collect()[0][0] or FACT_PLAYER_SEASON_ID_BASE

    new_rows = fact_player_season_new.join(
        existing.select("season_id", "player_id", "team_id"),
        on=["season_id", "player_id", "team_id"],
        how="left_anti"
    )

    if new_rows.count() > 0:
        new_rows = new_rows.withColumn(
            "fact_player_season_id",
            max_id + F.row_number().over(Window.orderBy("season_id", "player_id", "team_id"))
        )
        delta_table = DeltaTable.forPath(spark, fact_player_season_path)
        (
            delta_table.alias("t")
            .merge(
                new_rows.alias("s"),
                "t.season_id = s.season_id AND t.player_id = s.player_id AND t.team_id = s.team_id"
            )
            .whenNotMatchedInsertAll()
            .execute()
        )
        fact_player_season = spark.read.format("delta").load(fact_player_season_path)
    else:
        fact_player_season = existing
else:
    fact_player_season = fact_player_season_new.withColumn(
        "fact_player_season_id",
        FACT_PLAYER_SEASON_ID_BASE + F.row_number().over(Window.orderBy("season_id", "player_id", "team_id"))
    )
    fact_player_season.write.format("delta").mode("overwrite").save(fact_player_season_path)

display(fact_player_season)

## fact_standing

In [0]:
from pyspark.sql import functions as F
from datetime import date

fact_standing_path = "s3://football-data-raw-01/silver/fact_standing"
dim_team_path = "s3://football-data-raw-01/silver/dim_team"
dim_season_path = "s3://football-data-raw-01/silver/dim_season"

football_data_org_standings_paths = [
    f"s3://football-data-raw-01/raw/{year}/football_data_org/standings.json"
    for year in range(PROJECT_START_SEASON_YEAR, date.today().year + 1)
]
football_data_org_standings_paths = [p for p in football_data_org_standings_paths if path_exists(p)]

df_football_data_org_standings = spark.read.option("multiLine", "true").json(football_data_org_standings_paths)
dim_team = spark.read.format("delta").load(dim_team_path)
dim_season = spark.read.format("delta").load(dim_season_path)

df_football_data_org_stagings = (
    df_football_data_org_standings
    .select(F.col("season"),F.explode(F.col("standings")).alias("standings"))
    .select(F.col("season.id").alias("football_data_org_season_id"),
            F.col("standings.type").alias("type"),
            F.explode(F.col("standings.table")).alias("table"))
    .select(
        F.col("football_data_org_season_id"),
        F.col("type"),
        F.col("table.position").alias("position"),
        F.col("table.team.id").alias("football_data_org_team_id"),
        F.col("table.team.tla").alias("team_tla"),
        F.col("table.playedGames").alias("played_games"),
        F.col("table.won").alias("won"),
        F.col("table.draw").alias("draw"),
        F.col("table.lost").alias("lost"),
        F.col("table.goalsFor").alias("goals_for"),
        F.col("table.goalsAgainst").alias("goals_against"),
        F.col("table.goalDifference").alias("goal_difference"),
        F.col("table.points").alias("points"))
)

fact_standing = (
    df_football_data_org_stagings
    .join(dim_team, on=["football_data_org_team_id"], how="inner")
    .join(dim_season, on=["football_data_org_season_id"], how="inner")
    .select(
        F.col("season_id"),
        F.col("team_id"),
        F.col("type"),
        F.col("position"),
        F.col("played_games"),
        F.col("won"),
        F.col("draw"),
        F.col("lost"),
        F.col("goals_for"),
        F.col("goals_against"),
        F.col("goal_difference"),
        F.col("points"))
)

fact_standing.write.format("delta").mode("overwrite").save(fact_standing_path)

display(spark.read.format("delta").load(fact_standing_path))

## fact_match

In [0]:
from pyspark.sql import functions as F, Window
from delta.tables import DeltaTable
from datetime import date

FACT_MATCH_ID_BASE = 400000
fact_match_path = "s3://football-data-raw-01/silver/fact_match"
dim_team_path = "s3://football-data-raw-01/silver/dim_team"
dim_season_path = "s3://football-data-raw-01/silver/dim_season"

football_data_org_matches_paths = [
    f"s3://football-data-raw-01/raw/{year}/football_data_org/matches.json"
    for year in range(PROJECT_START_SEASON_YEAR, date.today().year + 1)
]
football_data_org_matches_paths = [p for p in football_data_org_matches_paths if path_exists(p)]
df_football_data_org_matches = spark.read.option("multiLine", "true").json(football_data_org_matches_paths)

soccerdata_fbref_schedule_paths = [
    f"s3://football-data-raw-01/raw/{year}/soccerdata_fbref/schedule.csv"
    for year in range(PROJECT_START_SEASON_YEAR, date.today().year + 1)
]
soccerdata_fbref_schedule_paths = [p for p in soccerdata_fbref_schedule_paths if path_exists(p)]
df_soccerdata_fbref_schedule = spark.read.option("inferSchema", "true").option("header", "true").csv(soccerdata_fbref_schedule_paths)

sports_api_pro_match_paths = []
for year in range(PROJECT_START_SEASON_YEAR, date.today().year + 1):
    folder = f"s3://football-data-raw-01/raw/{year}/sports_api_pro/"
    if path_exists(folder):
        sports_api_pro_match_paths += [
            f.path for f in dbutils.fs.ls(folder)
            if f.name.startswith("events_") and f.name.endswith(".json")
        ]
df_sports_api_pro_match = spark.read.option("multiLine", "true").json(sports_api_pro_match_paths)

dim_team = spark.read.format("delta").load(dim_team_path)
dim_season = spark.read.format("delta").load(dim_season_path)

df_football_data_org_matches = (
    df_football_data_org_matches
    .select(F.explode(F.col("matches")).alias("matches"))
    .select(F.col("matches.season.id").alias("football_data_org_season_id"),
            F.col("matches.id").alias("football_data_org_match_id"),
            F.col("matches.utcDate").alias("match_date"),
            F.col("matches.matchday").alias("gameweek"),
            F.col("matches.referees.name").alias("match_referee"),
            F.col("matches.homeTeam.id").alias("football_data_org_home_team_id"),
            F.col("matches.homeTeam.tla").alias("football_data_org_home_team_tla"),
            F.col("matches.homeTeam.shortName").alias("home_team_name"),
            F.col("matches.awayTeam.id").alias("football_data_org_away_team_id"),
            F.col("matches.awayTeam.shortName").alias("away_team_name"),
            F.col("matches.awayTeam.tla").alias("football_data_org_away_team_tla"),
            F.col("matches.score.fullTime.home").alias("full_time_home_team_score"),
            F.col("matches.score.fullTime.away").alias("full_time_away_team_score"),
            F.col("matches.score.halfTime.home").alias("half_time_home_team_score"),
            F.col("matches.score.halfTime.away").alias("half_time_away_team_score"),
    )
    .withColumn(
    "winner",
    F.when(
        F.col("full_time_home_team_score") > F.col("full_time_away_team_score"),
        F.col("football_data_org_home_team_tla")
    )
    .when(
        F.col("full_time_home_team_score") < F.col("full_time_away_team_score"),
        F.col("football_data_org_away_team_tla")
    )
    .otherwise(F.lit("DRAW")))
)

df_soccerdata_fbref_schedule = (
    df_soccerdata_fbref_schedule
    .select(F.col("season").alias("fbref_season_id"),
            F.col("date"),
            F.col("home_team"),
            F.col("away_team"),
            F.col("game_id").alias("fbref_match_id"),
            F.col("game").alias("fbref_game"))
)

df_sports_api_pro_match = (
    df_sports_api_pro_match
    .select(F.explode(F.col("data.events")).alias("col"))
    .select(F.col("col.season.id").alias("sports_api_pro_season_id"),
            F.col("col.roundInfo.round").alias("gameweek"),
            F.col("col.homeTeam.name").alias("home_team_name"),
            F.col("col.homeTeam.id").alias("sports_api_home_team_id"),
            F.col("col.awayTeam.name").alias("away_team_name"),
            F.col("col.awayTeam.id").alias("sports_api_away_team_id"),
            F.col("col.id").alias("sports_api_pro_match_id"),
            F.col("col.time.injuryTime1").alias("first_half_injury_time"),
            F.col("col.time.injuryTime2").alias("second_half_injury_time"),
            F.col("col.homeRedCards").alias("home_team_red_cards"),
            F.col("col.awayRedCards").alias("away_team_red_cards"),
            F.col("col.startTimestamp").alias("timestamp"))
    .withColumn(
        "match_date",
        F.timestamp_seconds(F.col("timestamp"))
    )
    .drop("timestamp")
    )

df_football_data_org_matches = df_football_data_org_matches.withColumn(
    "match_date_only", F.to_date(F.col("match_date"))
)

fact_match_new = (
    df_football_data_org_matches.alias("fd")
    .join(
        dim_team.alias("home_map"),
        F.col("fd.football_data_org_home_team_id") == F.col("home_map.football_data_org_team_id"),
        "left"
    )
    .join(
        dim_team.alias("away_map"),
        F.col("fd.football_data_org_away_team_id") == F.col("away_map.football_data_org_team_id"),
        "left"
    )
    .join(
        df_soccerdata_fbref_schedule.alias("fb"),
        (F.col("fd.match_date_only") == F.col("fb.date")) &
        (
            F.col("home_map.fbref_team_name").contains(F.col("fb.home_team")) |
            F.col("fb.home_team").contains(F.col("home_map.fbref_team_name"))
        ) &
        (
            F.col("away_map.fbref_team_name").contains(F.col("fb.away_team")) |
            F.col("fb.away_team").contains(F.col("away_map.fbref_team_name"))
        ),
        "left"
    )
    .select(
        "fd.*",
        F.col("fb.fbref_season_id"),
        F.col("fb.fbref_match_id"),
        F.col("fb.fbref_game"),
    )
    .drop("match_date_only")
)

fact_match_new = (
    fact_match_new.alias("fm")
    .join(
        dim_team.alias("home_map"),
        F.col("fm.football_data_org_home_team_id") == F.col("home_map.football_data_org_team_id"),
        "left"
    )
    .join(
        dim_team.alias("away_map"),
        F.col("fm.football_data_org_away_team_id") == F.col("away_map.football_data_org_team_id"),
        "left"
    )
    .join(
        df_sports_api_pro_match.alias("sp"),
        (F.col("home_map.sports_api_pro_team_id") == F.col("sp.sports_api_home_team_id")) &
        (F.col("away_map.sports_api_pro_team_id") == F.col("sp.sports_api_away_team_id")) &
        (F.to_date(F.col("fm.match_date")) == F.to_date(F.col("sp.match_date"))),
        "left"
    )
    .select(
        "fm.*",
        F.col("sp.sports_api_pro_match_id"),
        F.col("sp.sports_api_pro_season_id"),
        F.col("sp.first_half_injury_time"),
        F.col("sp.second_half_injury_time"),
        F.col("sp.home_team_red_cards"),
        F.col("sp.away_team_red_cards")
    )
)

fact_match_new = (
    fact_match_new.alias("fm")
    .join(
        dim_season.alias("ds"),
        F.col("fm.football_data_org_season_id") == F.col("ds.football_data_org_season_id"),
        "inner"
    )
    .join(
        dim_team.alias("ds_home"),
        F.col("fm.football_data_org_home_team_id") == F.col("ds_home.football_data_org_team_id"),
        "inner"
    )
    .join(
        dim_team.alias("ds_away"),
        F.col("fm.football_data_org_away_team_id") == F.col("ds_away.football_data_org_team_id"),
        "inner"
    )
    .select(
        "fm.*",
        F.col("ds.season_id").alias("season_id"),
        F.col("ds_home.team_id").alias("home_team_id"),
        F.col("ds_away.team_id").alias("away_team_id")
    )
    .drop(
        "football_data_org_season_id",
        "sports_api_pro_season_id",
        "football_data_org_home_team_id",
        "football_data_org_away_team_id",
        "fbref_season_id",
        "football_data_org_home_team_tla",
        "football_data_org_away_team_tla",
        "home_team_name",
        "away_team_name"
    )
    )

if DeltaTable.isDeltaTable(spark, fact_match_path):
    existing = spark.read.format("delta").load(fact_match_path)
    max_id = existing.agg(F.max("fact_match_id")).collect()[0][0] or FACT_MATCH_ID_BASE

    new_rows = fact_match_new.join(
        existing.select("football_data_org_match_id"),
        on="football_data_org_match_id",
        how="left_anti"
    )

    if new_rows.count() > 0:
        new_rows = new_rows.withColumn(
            "fact_match_id",
            max_id + F.row_number().over(Window.orderBy("football_data_org_match_id"))
        )
        delta_table = DeltaTable.forPath(spark, fact_match_path)
        (
            delta_table.alias("t")
            .merge(new_rows.alias("s"), "t.football_data_org_match_id = s.football_data_org_match_id")
            .whenNotMatchedInsertAll()
            .execute()
        )

    fact_match_with_id = fact_match_new.join(
        existing.select("football_data_org_match_id", "fact_match_id"),
        on="football_data_org_match_id",
        how="inner"
    )

    delta_table = DeltaTable.forPath(spark, fact_match_path)
    (
        delta_table.alias("t")
        .merge(fact_match_with_id.alias("s"), "t.football_data_org_match_id = s.football_data_org_match_id")
        .whenMatchedUpdateAll()
        .execute()
    )

    fact_match = spark.read.format("delta").load(fact_match_path)
else:
    fact_match = fact_match_new.withColumn(
        "fact_match_id",
        FACT_MATCH_ID_BASE + F.row_number().over(Window.orderBy("football_data_org_match_id"))
    )
    fact_match.write.format("delta").mode("overwrite").save(fact_match_path)

display(fact_match)

## fact_match_events

In [0]:
from pyspark.sql import functions as F, Window
from functools import reduce
from delta.tables import DeltaTable
from datetime import date

EVENT_ID_BASE = 500000
fact_match_events_path = "s3://football-data-raw-01/silver/fact_match_events"
fact_match_path = "s3://football-data-raw-01/silver/fact_match"
dim_season_path = "s3://football-data-raw-01/silver/dim_season"
dim_team_path = "s3://football-data-raw-01/silver/dim_team"
dim_players_path = "s3://football-data-raw-01/silver/dim_players"

soccerdata_fbref_events_paths = [
    f"s3://football-data-raw-01/raw/{year}/soccerdata_fbref/events.csv"
    for year in range(PROJECT_START_SEASON_YEAR, date.today().year + 1)
]
soccerdata_fbref_events_paths = [p for p in soccerdata_fbref_events_paths if path_exists(p)]
df_soccerdata_fbref_events = spark.read.option("inferSchema", "true").option("header", "true").csv(soccerdata_fbref_events_paths)

fact_match = spark.read.format("delta").load(fact_match_path)
dim_season = spark.read.format("delta").load(dim_season_path)
dim_team = spark.read.format("delta").load(dim_team_path)
dim_players = spark.read.format("delta").load(dim_players_path)


dim_match_events = (
    df_soccerdata_fbref_events.alias("fs")
    .join(
        fact_match.alias("fm"),
        F.col("fs.game") == F.col("fm.fbref_game"),
        "left"
        )
    .select(
        F.col("fs.season").alias("fbref_season_id"),
        F.col("fs.team"),
        F.col("fs.minute"),
        F.col("fs.player1"),
        F.col("fs.player2"),
        F.col("fs.event_type"),
        F.col("fm.fbref_match_id"),
        F.col("fs.game"),
    )
    .withColumn("game_teams", F.regexp_extract(F.col("game"), r"^\d{4}-\d{2}-\d{2}\s(.+)$", 1))
    .withColumn("home_team_raw", F.regexp_extract(F.col("game_teams"), r"^(.+)-(.+)$", 1))
    .withColumn("away_team_raw", F.regexp_extract(F.col("game_teams"), r"^(.+)-(.+)$", 2))
    .withColumn(
        "derived_team",
        F.when(
            F.col("home_team_raw").contains(F.col("team")) | F.col("team").contains(F.col("home_team_raw")),
            F.col("home_team_raw")
        ).otherwise(F.col("away_team_raw"))
    )
    .drop("game", "game_teams", "home_team_raw", "away_team_raw")
    )



fact_match_events_new = (
    dim_match_events.alias("dme")
    .join(
        dim_season.alias("ds"),
        F.col("dme.fbref_season_id") == F.col("ds.fbref_season_id"),
        "inner"
        )
    .join(
        dim_team.alias("dt"),
        F.col("dme.derived_team") == F.col("dt.fbref_team_name"),
        "inner"
    )
    .join(
        fact_match.alias("fm"),
        F.col("fm.fbref_match_id") == F.col("dme.fbref_match_id"),
        "inner"
        )
    .join(
        dim_players.alias("dp_1"),
        F.col("dme.player1") == F.col("dp_1.player_name"),
        "left"
    )
    .join(
        dim_players.alias("dp_2"),
        F.col("dme.player2") == F.col("dp_2.player_name"),
        "left"
    )
    .select(
        F.col("dme.*"),
        F.col("dt.team_id").alias("team_id"),
        F.col("ds.season_id").alias("season_id"),
        F.col("fm.fact_match_id").alias("match_id"),
        F.col("dp_1.player_id").alias("player_1_id"),
        F.col("dp_2.player_id").alias("player_2_id")
    )
    .drop(
        "fbref_season_id",
        "team",
        "fbref_match_id",
        "player1",
        "player2"
    )
    )
event_key_cols = ["match_id", "minute", "event_type", "team_id", "player_1_id", "player_2_id"]

if DeltaTable.isDeltaTable(spark, fact_match_events_path):
    existing = spark.read.format("delta").load(fact_match_events_path)
    max_id = existing.agg(F.max("event_id")).collect()[0][0] or EVENT_ID_BASE

    join_condition = reduce(
        lambda acc, c: acc & F.col(f"new.{c}").eqNullSafe(F.col(f"exist.{c}")),
        event_key_cols[1:],
        F.col(f"new.{event_key_cols[0]}").eqNullSafe(F.col(f"exist.{event_key_cols[0]}"))
    )

    new_rows = (
        fact_match_events_new.alias("new")
        .join(existing.select(*event_key_cols).alias("exist"), join_condition, "left_anti")
    )

    if new_rows.count() > 0:
        new_rows = new_rows.withColumn(
            "event_id",
            max_id + F.row_number().over(Window.orderBy(*event_key_cols))
        )
        delta_table = DeltaTable.forPath(spark, fact_match_events_path)
        merge_condition = " AND ".join([f"t.{c} <=> s.{c}" for c in event_key_cols])
        (
            delta_table.alias("t")
            .merge(new_rows.alias("s"), merge_condition)
            .whenNotMatchedInsertAll()
            .execute()
        )
        fact_match_events = spark.read.format("delta").load(fact_match_events_path)
    else:
        fact_match_events = existing
else:
    fact_match_events = fact_match_events_new.withColumn(
        "event_id",
        EVENT_ID_BASE + F.row_number().over(Window.orderBy(*event_key_cols))
    )
    fact_match_events.write.format("delta").mode("overwrite").save(fact_match_events_path)

display(fact_match_events)

## fact_match_lineups

In [0]:
from pyspark.sql import functions as F, Window
from functools import reduce
from delta.tables import DeltaTable
from datetime import date

LINEUP_ID_BASE = 600000
fact_match_lineups_path = "s3://football-data-raw-01/silver/fact_match_lineups"
fact_match_path = "s3://football-data-raw-01/silver/fact_match"
dim_season_path = "s3://football-data-raw-01/silver/dim_season"
dim_team_path = "s3://football-data-raw-01/silver/dim_team"
dim_players_path = "s3://football-data-raw-01/silver/dim_players"

soccerdata_fbref_lineup_paths = [
    f"s3://football-data-raw-01/raw/{year}/soccerdata_fbref/lineup.csv"
    for year in range(PROJECT_START_SEASON_YEAR, date.today().year + 1)
]
soccerdata_fbref_lineup_paths = [p for p in soccerdata_fbref_lineup_paths if path_exists(p)]
df_soccerdata_fbref_lineup = spark.read.option("inferSchema", "true").option("header", "true").csv(soccerdata_fbref_lineup_paths)

fact_match = spark.read.format("delta").load(fact_match_path)
dim_season = spark.read.format("delta").load(dim_season_path)
dim_team = spark.read.format("delta").load(dim_team_path)
dim_players = spark.read.format("delta").load(dim_players_path)

dim_match_lineups = (
    df_soccerdata_fbref_lineup.alias("fs")
    .join(
        fact_match.alias("fm"),
        F.col("fs.game") == F.col("fm.fbref_game"),
        "left"
        )
    .select(
        F.col("fs.season").alias("fbref_season_id"),
        F.col("fs.team"),
        F.col("fs.player"),
        F.col("fs.jersey_number"),
        F.col("fs.position"),
        F.col("fs.is_starter"),
        F.col("fs.minutes_played"),
        F.col("fm.fbref_match_id"),
        F.col("fs.game")
    )
    .withColumn("game_teams", F.regexp_extract(F.col("game"), r"^\d{4}-\d{2}-\d{2}\s(.+)$", 1))
    .withColumn("home_team_raw", F.regexp_extract(F.col("game_teams"), r"^(.+)-(.+)$", 1))
    .withColumn("away_team_raw", F.regexp_extract(F.col("game_teams"), r"^(.+)-(.+)$", 2))
    .withColumn(
        "derived_team",
        F.when(
            F.col("home_team_raw").contains(F.col("team")) | F.col("team").contains(F.col("home_team_raw")),
            F.col("home_team_raw")
        ).otherwise(F.col("away_team_raw"))
    )
    .drop("game", "game_teams", "home_team_raw", "away_team_raw")
    )

fact_match_lineups_new = (
    dim_match_lineups.alias("dml")
    .join(
        dim_season.alias("ds"),
        F.col("dml.fbref_season_id") == F.col("ds.fbref_season_id"),
        "inner"
        )
    .join(
        dim_team.alias("dt"),
        F.col("dml.derived_team") == F.col("dt.fbref_team_name"),
        "inner"
    )
    .join(
        fact_match.alias("fm"),
        F.col("fm.fbref_match_id") == F.col("dml.fbref_match_id"),
        "inner"
        )
    .join(
        dim_players.alias("dp"),
        F.col("dml.player") == F.col("dp.player_name"),
        "left"
    )
    .select(
        F.col("dml.*"),
        F.col("dt.team_id").alias("team_id"),
        F.col("ds.season_id").alias("season_id"),
        F.col("fm.fact_match_id").alias("match_id"),
        F.col("dp.player_id").alias("player_id")
    )
    .drop(
        "fbref_season_id",
        "team",
        "fbref_match_id",
        "player"
    )
    )

lineup_key_cols = ["match_id", "team_id", "player_id"]

if DeltaTable.isDeltaTable(spark, fact_match_lineups_path):
    existing = spark.read.format("delta").load(fact_match_lineups_path)
    max_id = existing.agg(F.max("lineup_id")).collect()[0][0] or LINEUP_ID_BASE

    join_condition = reduce(
        lambda acc, c: acc & F.col(f"new.{c}").eqNullSafe(F.col(f"exist.{c}")),
        lineup_key_cols[1:],
        F.col(f"new.{lineup_key_cols[0]}").eqNullSafe(F.col(f"exist.{lineup_key_cols[0]}"))
    )

    new_rows = (
        fact_match_lineups_new.alias("new")
        .join(existing.select(*lineup_key_cols).alias("exist"), join_condition, "left_anti")
    )

    if new_rows.count() > 0:
        new_rows = new_rows.withColumn(
            "lineup_id",
            max_id + F.row_number().over(Window.orderBy(*lineup_key_cols))
        )
        delta_table = DeltaTable.forPath(spark, fact_match_lineups_path)
        merge_condition = " AND ".join([f"t.{c} <=> s.{c}" for c in lineup_key_cols])
        (
            delta_table.alias("t")
            .merge(new_rows.alias("s"), merge_condition)
            .whenNotMatchedInsertAll()
            .execute()
        )
        fact_match_lineups = spark.read.format("delta").load(fact_match_lineups_path)
    else:
        fact_match_lineups = existing
else:
    fact_match_lineups = fact_match_lineups_new.withColumn(
        "lineup_id",
        LINEUP_ID_BASE + F.row_number().over(Window.orderBy(*lineup_key_cols))
    )
    fact_match_lineups.write.format("delta").mode("overwrite").save(fact_match_lineups_path)

display(fact_match_lineups)

## fact_team_season_standard_stats

In [0]:
from pyspark.sql import functions as F
from datetime import date

soccerdata_fbref_team_season_standard_stats_paths = [
    f"s3://football-data-raw-01/raw/{year}/soccerdata_fbref/team_season_standard_stats.csv"
    for year in range(PROJECT_START_SEASON_YEAR, date.today().year + 1)
]
soccerdata_fbref_team_season_standard_stats_paths = [p for p in soccerdata_fbref_team_season_standard_stats_paths if path_exists(p)]
fact_team_season_standard_stats_path = "s3://football-data-raw-01/silver/fact_team_season_standard_stats"

df_soccerdata_fbref_team_season_standard_stats = reduce(
    lambda a, b: a.unionByName(b),
    [flatten_two_row_header(p) for p in soccerdata_fbref_team_season_standard_stats_paths]
)

dim_team = spark.read.format("delta").load("s3://football-data-raw-01/silver/dim_team")
dim_season = spark.read.format("delta").load("s3://football-data-raw-01/silver/dim_season")

df_soccerdata_fbref_team_season_standard_stats = (
    df_soccerdata_fbref_team_season_standard_stats
    .withColumnRenamed("season", "fbref_season_id")
)

fact_team_season_standard_stats = (
    df_soccerdata_fbref_team_season_standard_stats.alias("fs")
    .join(
        dim_team.alias("t"),
        F.expr("fs.team = t.fbref_team_name"),
        "inner"
    )
    .join(
        dim_season.alias("s"),
        F.expr("fs.fbref_season_id = s.fbref_season_id"),
        "inner"
    )
    .select(
        F.col("fs.*"),
        F.col("t.team_id"),
        F.col("s.season_id")
    )
    .drop("league","url","fbref_season_id","team","fbref_team_id")
)
fact_team_season_standard_stats = sanitize_columns(fact_team_season_standard_stats)
fact_team_season_standard_stats.write.format("delta").mode("overwrite").save(fact_team_season_standard_stats_path)

display(spark.read.format("delta").load(fact_team_season_standard_stats_path))

## fact_team_season_stats_shooting


In [0]:
from pyspark.sql import functions as F
from functools import reduce
from datetime import date

soccerdata_fbref_team_season_shooting_stats_paths = [
    f"s3://football-data-raw-01/raw/{year}/soccerdata_fbref/team_season_shooting_stats.csv"
    for year in range(PROJECT_START_SEASON_YEAR, date.today().year + 1)
]
soccerdata_fbref_team_season_shooting_stats_paths = [p for p in soccerdata_fbref_team_season_shooting_stats_paths if path_exists(p)]
fact_team_season_shooting_stats_path = "s3://football-data-raw-01/silver/fact_team_season_shooting_stats"

df_soccerdata_fbref_team_season_shooting_stats = reduce(
    lambda a, b: a.unionByName(b),
    [flatten_two_row_header(p) for p in soccerdata_fbref_team_season_shooting_stats_paths]
)

dim_team = spark.read.format("delta").load("s3://football-data-raw-01/silver/dim_team")
dim_season = spark.read.format("delta").load("s3://football-data-raw-01/silver/dim_season")

df_soccerdata_fbref_team_season_shooting_stats = (
    df_soccerdata_fbref_team_season_shooting_stats
    .withColumnRenamed("season", "fbref_season_id")
)

fact_team_season_shooting_stats = (
    df_soccerdata_fbref_team_season_shooting_stats.alias("fs")
    .join(
        dim_team.alias("t"),
        F.expr("fs.team = t.fbref_team_name"),
        "inner"
    )
    .join(
        dim_season.alias("s"),
        F.expr("fs.fbref_season_id = s.fbref_season_id"),
        "inner"
    )
    .select(
        F.col("fs.*"),
        F.col("t.team_id"),
        F.col("s.season_id")
    )
    .drop("league","url","fbref_season_id","team")
)

fact_team_season_shooting_stats = sanitize_columns(fact_team_season_shooting_stats)
fact_team_season_shooting_stats.write.format("delta").mode("overwrite").save(fact_team_season_shooting_stats_path)

display(spark.read.format("delta").load(fact_team_season_shooting_stats_path))

## fact_team_season_playing_time_stats

In [0]:
from pyspark.sql import functions as F
from functools import reduce
from datetime import date

soccerdata_fbref_team_season_playing_time_stats_paths = [
    f"s3://football-data-raw-01/raw/{year}/soccerdata_fbref/team_season_playing_time_stats.csv"
    for year in range(PROJECT_START_SEASON_YEAR, date.today().year + 1)
]
soccerdata_fbref_team_season_playing_time_stats_paths = [p for p in soccerdata_fbref_team_season_playing_time_stats_paths if path_exists(p)]
fact_team_season_playing_time_stats_path = "s3://football-data-raw-01/silver/fact_team_season_playing_time_stats"

df_soccerdata_fbref_team_season_playing_time_stats = reduce(
    lambda a, b: a.unionByName(b),
    [flatten_two_row_header(p) for p in soccerdata_fbref_team_season_playing_time_stats_paths]
)

dim_team = spark.read.format("delta").load("s3://football-data-raw-01/silver/dim_team")
dim_season = spark.read.format("delta").load("s3://football-data-raw-01/silver/dim_season")

df_soccerdata_fbref_team_season_playing_time_stats = (
    df_soccerdata_fbref_team_season_playing_time_stats
    .withColumnRenamed("season", "fbref_season_id")
)

fact_team_season_playing_time_stats = (
    df_soccerdata_fbref_team_season_playing_time_stats.alias("fs")
    .join(
        dim_team.alias("t"),
        F.expr("fs.team = t.fbref_team_name"),
        "inner"
    )
    .join(
        dim_season.alias("s"),
        F.expr("fs.fbref_season_id = s.fbref_season_id"),
        "inner"
    )
    .select(
        F.col("fs.*"),
        F.col("t.team_id"),
        F.col("s.season_id")
    )
    .drop("league","url","fbref_season_id","team")
)

fact_team_season_playing_time_stats = sanitize_columns(fact_team_season_playing_time_stats)
fact_team_season_playing_time_stats.write.format("delta").mode("overwrite").save(fact_team_season_playing_time_stats_path)

display(spark.read.format("delta").load(fact_team_season_playing_time_stats_path))

## fact_team_season_keeper_stats

In [0]:
from pyspark.sql import functions as F
from functools import reduce
from datetime import date

soccerdata_fbref_team_season_keeper_stats_paths = [
    f"s3://football-data-raw-01/raw/{year}/soccerdata_fbref/team_season_keeper_stats.csv"
    for year in range(PROJECT_START_SEASON_YEAR, date.today().year + 1)
]
soccerdata_fbref_team_season_keeper_stats_paths = [p for p in soccerdata_fbref_team_season_keeper_stats_paths if path_exists(p)]
fact_team_season_keeper_stats_path = "s3://football-data-raw-01/silver/fact_team_season_keeper_stats"

df_soccerdata_fbref_team_season_keeper_stats = reduce(
    lambda a, b: a.unionByName(b),
    [flatten_two_row_header(p) for p in soccerdata_fbref_team_season_keeper_stats_paths]
)

dim_team = spark.read.format("delta").load("s3://football-data-raw-01/silver/dim_team")
dim_season = spark.read.format("delta").load("s3://football-data-raw-01/silver/dim_season")

df_soccerdata_fbref_team_season_keeper_stats = (
    df_soccerdata_fbref_team_season_keeper_stats
    .withColumnRenamed("season", "fbref_season_id")
)

fact_team_season_keeper_stats = (
    df_soccerdata_fbref_team_season_keeper_stats.alias("fs")
    .join(
        dim_team.alias("t"),
        F.expr("fs.team = t.fbref_team_name"),
        "inner"
    )
    .join(
        dim_season.alias("s"),
        F.expr("fs.fbref_season_id = s.fbref_season_id"),
        "inner"
    )
    .select(
        F.col("fs.*"),
        F.col("t.team_id"),
        F.col("s.season_id")
    )
    .drop("league","url","fbref_season_id","team")
)

fact_team_season_keeper_stats = sanitize_columns(fact_team_season_keeper_stats)
fact_team_season_keeper_stats.write.format("delta").mode("overwrite").save(fact_team_season_keeper_stats_path)

display(spark.read.format("delta").load(fact_team_season_keeper_stats_path))

## fact_team_season_misc_stats

In [0]:
from pyspark.sql import functions as F
from functools import reduce
from datetime import date

soccerdata_fbref_team_season_misc_stats_paths = [
    f"s3://football-data-raw-01/raw/{year}/soccerdata_fbref/team_season_misc_stats.csv"
    for year in range(PROJECT_START_SEASON_YEAR, date.today().year + 1)
]
soccerdata_fbref_team_season_misc_stats_paths = [p for p in soccerdata_fbref_team_season_misc_stats_paths if path_exists(p)]
fact_team_season_misc_stats_path = "s3://football-data-raw-01/silver/fact_team_season_misc_stats"

df_soccerdata_fbref_team_season_misc_stats = reduce(
    lambda a, b: a.unionByName(b),
    [flatten_two_row_header(p) for p in soccerdata_fbref_team_season_misc_stats_paths]
)

dim_team = spark.read.format("delta").load("s3://football-data-raw-01/silver/dim_team")
dim_season = spark.read.format("delta").load("s3://football-data-raw-01/silver/dim_season")

df_soccerdata_fbref_team_season_misc_stats = (
    df_soccerdata_fbref_team_season_misc_stats
    .withColumnRenamed("season", "fbref_season_id")
)

fact_team_season_misc_stats = (
    df_soccerdata_fbref_team_season_misc_stats.alias("fs")
    .join(
        dim_team.alias("t"),
        F.expr("fs.team = t.fbref_team_name"),
        "inner"
    )
    .join(
        dim_season.alias("s"),
        F.expr("fs.fbref_season_id = s.fbref_season_id"),
        "inner"
    )
    .select(
        F.col("fs.*"),
        F.col("t.team_id"),
        F.col("s.season_id")
    )
    .drop("league","url","fbref_season_id","team","Performance_PKwon","Performance_PKcom")
)

fact_team_season_misc_stats = sanitize_columns(fact_team_season_misc_stats)
fact_team_season_misc_stats.write.format("delta").mode("overwrite").save(fact_team_season_misc_stats_path)

display(spark.read.format("delta").load(fact_team_season_misc_stats_path))

## fact_team_match_standard_stats

In [0]:
from pyspark.sql import functions as F
from datetime import date

soccerdata_fbref_team_match_schedule_stats_paths = [
    f"s3://football-data-raw-01/raw/{year}/soccerdata_fbref/team_match_schedule_stats.csv"
    for year in range(PROJECT_START_SEASON_YEAR, date.today().year + 1)
]
soccerdata_fbref_team_match_schedule_stats_paths = [p for p in soccerdata_fbref_team_match_schedule_stats_paths if path_exists(p)]
fact_team_match_standard_stats_path = "s3://football-data-raw-01/silver/fact_team_match_standard_stats"

df_soccerdata_fbref_team_match_schedule_stats = spark.read.option("inferSchema", "true").option("header", "true").csv(soccerdata_fbref_team_match_schedule_stats_paths)

dim_players = spark.read.format("delta").load("s3://football-data-raw-01/silver/dim_players")
fact_match = spark.read.format("delta").load("s3://football-data-raw-01/silver/fact_match")

fact_team_match_standard_stats = (
    df_soccerdata_fbref_team_match_schedule_stats
    .withColumnRenamed("season", "fbref_season_id")
    .select(
        F.col("fbref_season_id"),
        F.col("opponent"),
        F.col("venue"),
        F.col("Poss"),
        F.col("Captain"),
        F.col("Formation"),
        F.col("Opp Formation"),
        F.col("match_report")
    )
    .withColumn(
        "fbref_match_id",
        F.regexp_extract(F.col("match_report"), r"/matches/([a-f0-9]+)/", 1)
    )
    .drop("match_report")
)

fact_team_match_standard_stats = (
    fact_team_match_standard_stats.alias("f")
    .join(
        dim_players.alias("p"),
        F.expr("p.player_name = f.Captain"),
        "inner"
    )
    .join(
        fact_match.alias("m"),
        F.expr("m.fbref_match_id = f.fbref_match_id"),
        "inner"
    )
    .select(
        F.col("m.season_id"),
        F.col("p.player_id").alias("captain_id"),
        F.col("m.fact_match_id").alias("match_id"),
        # The raw `team` column is empty, so the row's own team comes from venue.
        # (Joining dim_team on `opponent` gave every row its opponent's team_id.)
        F.when(F.col("f.venue") == "Home", F.col("m.home_team_id"))
        .otherwise(F.col("m.away_team_id")).alias("team_id"),
        F.col("Formation"),
        F.col("Opp Formation"),
        F.col("Poss")
    )
  
)

fact_team_match_standard_stats = sanitize_columns(fact_team_match_standard_stats)
fact_team_match_standard_stats.write.format("delta").mode("overwrite").save(fact_team_match_standard_stats_path)

display(spark.read.format("delta").load(fact_team_match_standard_stats_path))

## fact_team_match_shooting_stats

In [0]:
from pyspark.sql import functions as F
from functools import reduce
from datetime import date

soccerdata_fbref_team_match_shooting_stats_paths = [
    f"s3://football-data-raw-01/raw/{year}/soccerdata_fbref/team_match_shooting_stats.csv"
    for year in range(PROJECT_START_SEASON_YEAR, date.today().year + 1)
]
soccerdata_fbref_team_match_shooting_stats_paths = [p for p in soccerdata_fbref_team_match_shooting_stats_paths if path_exists(p)]
fact_team_match_shooting_stats_path = "s3://football-data-raw-01/silver/fact_team_match_shooting_stats"

df_soccerdata_fbref_team_match_shooting_stats = reduce(
    lambda a, b: a.unionByName(b),
    [flatten_two_row_header(p) for p in soccerdata_fbref_team_match_shooting_stats_paths]
)

fact_match = spark.read.format("delta").load("s3://football-data-raw-01/silver/fact_match")

fact_team_match_shooting_stats = (
    df_soccerdata_fbref_team_match_shooting_stats
    .withColumnRenamed("season", "fbref_season_id")
    .drop("league","team","game","date","round","day","result","GF","GA","time")
    .withColumn(
        "fbref_match_id",
        F.regexp_extract(F.col("match_report"), r"/matches/([a-f0-9]+)/", 1)
    )
    .drop("match_report")
)

fact_team_match_shooting_stats = (
    fact_team_match_shooting_stats.alias("f")
    .join(
        fact_match.alias("m"),
        F.expr("m.fbref_match_id = f.fbref_match_id"),
        "inner"
    )
    .select(
        F.col("f.*"),
        # The raw `team` column is empty, so the row's own team comes from venue.
        # (Joining dim_team on `opponent` gave every row its opponent's team_id.)
        F.when(F.col("f.venue") == "Home", F.col("m.home_team_id"))
        .otherwise(F.col("m.away_team_id")).alias("team_id"),
        F.col("m.season_id"),
        F.col("m.fact_match_id").alias("match_id")
    )
    .drop("opponent","venue","fbref_season_id","fbref_match_id")
)

fact_team_match_shooting_stats = sanitize_columns(fact_team_match_shooting_stats)
fact_team_match_shooting_stats.write.format("delta").mode("overwrite").save(fact_team_match_shooting_stats_path)

display(spark.read.format("delta").load(fact_team_match_shooting_stats_path))

## fact_team_match_misc_stats

In [0]:
from pyspark.sql import functions as F
from functools import reduce
from datetime import date

soccerdata_fbref_team_match_misc_stats_paths = [
    f"s3://football-data-raw-01/raw/{year}/soccerdata_fbref/team_match_misc_stats.csv"
    for year in range(PROJECT_START_SEASON_YEAR, date.today().year + 1)
]
soccerdata_fbref_team_match_misc_stats_paths = [p for p in soccerdata_fbref_team_match_misc_stats_paths if path_exists(p)]
fact_team_match_misc_stats_path = "s3://football-data-raw-01/silver/fact_team_match_misc_stats"

df_soccerdata_fbref_team_match_misc_stats = reduce(
    lambda a, b: a.unionByName(b),
    [flatten_two_row_header(p) for p in soccerdata_fbref_team_match_misc_stats_paths]
)

fact_match = spark.read.format("delta").load("s3://football-data-raw-01/silver/fact_match")

fact_team_match_misc_stats = (
    df_soccerdata_fbref_team_match_misc_stats
    .withColumnRenamed("season", "fbref_season_id")
    .drop("league","team","game","date","round","day","result","GF","GA","time","Performance_PKwon","Performance_PKcon")
    .withColumn(
        "fbref_match_id",
        F.regexp_extract(F.col("match_report"), r"/matches/([a-f0-9]+)/", 1)
    )
    .drop("match_report")
)

fact_team_match_misc_stats = (
    fact_team_match_misc_stats.alias("f")
    .join(
        fact_match.alias("m"),
        F.expr("m.fbref_match_id = f.fbref_match_id"),
        "inner"
    )
    .select(
        F.col("f.*"),
        # The raw `team` column is empty, so the row's own team comes from venue.
        # (Joining dim_team on `opponent` gave every row its opponent's team_id.)
        F.when(F.col("f.venue") == "Home", F.col("m.home_team_id"))
        .otherwise(F.col("m.away_team_id")).alias("team_id"),
        F.col("m.season_id"),
        F.col("m.fact_match_id").alias("match_id")
    )
    .drop("opponent","venue","fbref_season_id","fbref_match_id")
)

fact_team_match_misc_stats = sanitize_columns(fact_team_match_misc_stats)
fact_team_match_misc_stats.write.format("delta").mode("overwrite").save(fact_team_match_misc_stats_path)

display(spark.read.format("delta").load(fact_team_match_misc_stats_path))

## fact_team_match_keeper_stats

In [0]:
from pyspark.sql import functions as F
from functools import reduce
from datetime import date

soccerdata_fbref_team_match_keeper_stats_paths = [
    f"s3://football-data-raw-01/raw/{year}/soccerdata_fbref/team_match_keeper_stats.csv"
    for year in range(PROJECT_START_SEASON_YEAR, date.today().year + 1)
]
soccerdata_fbref_team_match_keeper_stats_paths = [p for p in soccerdata_fbref_team_match_keeper_stats_paths if path_exists(p)]
fact_team_match_keeper_stats_path = "s3://football-data-raw-01/silver/fact_team_match_keeper_stats"

df_soccerdata_fbref_team_match_keeper_stats = reduce(
    lambda a, b: a.unionByName(b),
    [flatten_two_row_header(p) for p in soccerdata_fbref_team_match_keeper_stats_paths]
)

fact_match = spark.read.format("delta").load("s3://football-data-raw-01/silver/fact_match")

fact_team_match_keeper_stats = (
    df_soccerdata_fbref_team_match_keeper_stats
    .withColumnRenamed("season", "fbref_season_id")
    .drop("league","team","game","date","round","day","result","GF","GA","time")
    .withColumn(
        "fbref_match_id",
        F.regexp_extract(F.col("match_report"), r"/matches/([a-f0-9]+)/", 1)
    )
    .drop("match_report")
)

fact_team_match_keeper_stats = (
    fact_team_match_keeper_stats.alias("f")
    .join(
        fact_match.alias("m"),
        F.expr("m.fbref_match_id = f.fbref_match_id"),
        "inner"
    )
    .select(
        F.col("f.*"),
        # The raw `team` column is empty, so the row's own team comes from venue.
        # (Joining dim_team on `opponent` gave every row its opponent's team_id.)
        F.when(F.col("f.venue") == "Home", F.col("m.home_team_id"))
        .otherwise(F.col("m.away_team_id")).alias("team_id"),
        F.col("m.season_id"),
        F.col("m.fact_match_id").alias("match_id")
    )
    .drop("opponent","venue","fbref_season_id","fbref_match_id")
)

fact_team_match_keeper_stats = sanitize_columns(fact_team_match_keeper_stats)
fact_team_match_keeper_stats.write.format("delta").mode("overwrite").save(fact_team_match_keeper_stats_path)

display(spark.read.format("delta").load(fact_team_match_keeper_stats_path))

## fact_player_match_stats_summary

In [0]:
from pyspark.sql import functions as F
from functools import reduce
from datetime import date

soccerdata_fbref_player_match_summary_stats_paths = [
    f"s3://football-data-raw-01/raw/{year}/soccerdata_fbref/player_match_stats_summary.csv"
    for year in range(PROJECT_START_SEASON_YEAR, date.today().year + 1)
]
soccerdata_fbref_player_match_summary_stats_paths = [p for p in soccerdata_fbref_player_match_summary_stats_paths if path_exists(p)]
fact_player_match_stats_summary_path = "s3://football-data-raw-01/silver/fact_player_match_stats_summary"

df_soccerdata_fbref_player_match_summary_stats = reduce(
    lambda a, b: a.unionByName(b),
    [flatten_two_row_header(p) for p in soccerdata_fbref_player_match_summary_stats_paths]
)

dim_team = spark.read.format("delta").load("s3://football-data-raw-01/silver/dim_team")
fact_match = spark.read.format("delta").load("s3://football-data-raw-01/silver/fact_match")
dim_players = spark.read.format("delta").load("s3://football-data-raw-01/silver/dim_players")

fact_player_match_stats_summary = (
    df_soccerdata_fbref_player_match_summary_stats
    .withColumnRenamed("season","fbref_season_id")
    .withColumnRenamed("game_id","fbref_match_id")
    .drop("Performance_PKwon","Performance_PKcon","league","nation","age","game")
)

fact_player_match_stats_summary = (
    fact_player_match_stats_summary.alias("f")
    .join(
        dim_team.alias("t"),
        F.expr("t.fbref_team_name = f.team"),
        "inner"
    )
    .join(
        fact_match.alias("m"),
        F.expr("m.fbref_match_id = f.fbref_match_id"),
        "inner"
    )
    .join(
        dim_players.alias("p"),
        F.expr("p.player_name = f.player"),
        "inner"
    )
    .select(
        F.col("f.*"),
        F.col("t.team_id"),
        F.col("m.season_id"),
        F.col("m.fact_match_id").alias("match_id"),
        F.col("p.player_id")
    )
    .drop("team","player","fbref_season_id","fbref_match_id")
)

fact_player_match_stats_summary = sanitize_columns(fact_player_match_stats_summary)
fact_player_match_stats_summary.write.format("delta").mode("overwrite").save(fact_player_match_stats_summary_path)

display(spark.read.format("delta").load(fact_player_match_stats_summary_path))

## fact_player_match_stats_keepers

In [0]:
from pyspark.sql import functions as F
from functools import reduce
from datetime import date

soccerdata_fbref_player_match_keepers_stats_paths = [
    f"s3://football-data-raw-01/raw/{year}/soccerdata_fbref/player_match_stats_keepers.csv"
    for year in range(PROJECT_START_SEASON_YEAR, date.today().year + 1)
]
soccerdata_fbref_player_match_keepers_stats_paths = [p for p in soccerdata_fbref_player_match_keepers_stats_paths if path_exists(p)]
fact_player_match_keepers_summary_path = "s3://football-data-raw-01/silver/fact_player_match_keepers_summary"

df_soccerdata_fbref_player_match_keepers_stats = reduce(
    lambda a, b: a.unionByName(b),
    [flatten_two_row_header(p) for p in soccerdata_fbref_player_match_keepers_stats_paths]
)

dim_team = spark.read.format("delta").load("s3://football-data-raw-01/silver/dim_team")
fact_match = spark.read.format("delta").load("s3://football-data-raw-01/silver/fact_match")
dim_players = spark.read.format("delta").load("s3://football-data-raw-01/silver/dim_players")

fact_player_match_keepers_summary = (
    df_soccerdata_fbref_player_match_keepers_stats
    .withColumnRenamed("season","fbref_season_id")
    .withColumnRenamed("game_id","fbref_match_id")
    .drop("league","nation","age","game")
)

fact_player_match_keepers_summary = (
    fact_player_match_keepers_summary.alias("f")
    .join(
        dim_team.alias("t"),
        F.expr("t.fbref_team_name = f.team"),
        "inner"
    )
    .join(
        fact_match.alias("m"),
        F.expr("m.fbref_match_id = f.fbref_match_id"),
        "inner"
    )
    .join(
        dim_players.alias("p"),
        F.expr("p.player_name = f.player"),
        "inner"
    )
    .select(
        F.col("f.*"),
        F.col("t.team_id"),
        F.col("m.season_id"),
        F.col("m.fact_match_id").alias("match_id"),
        F.col("p.player_id")
    )
    .drop("team","player","fbref_season_id","fbref_match_id")
)

fact_player_match_keepers_summary = sanitize_columns(fact_player_match_keepers_summary)
fact_player_match_keepers_summary.write.format("delta").mode("overwrite").save(fact_player_match_keepers_summary_path)

display(spark.read.format("delta").load(fact_player_match_keepers_summary_path))

## fact_players_season_standard_stats

In [0]:
from pyspark.sql import functions as F
from functools import reduce
from datetime import date

soccerdata_fbref_player_season_standard_stats_paths = [
    f"s3://football-data-raw-01/raw/{year}/soccerdata_fbref/players_season_standard_stats.csv"
    for year in range(PROJECT_START_SEASON_YEAR, date.today().year + 1)
]
soccerdata_fbref_player_season_standard_stats_paths = [p for p in soccerdata_fbref_player_season_standard_stats_paths if path_exists(p)]
fact_players_season_standard_stats_path = "s3://football-data-raw-01/silver/fact_players_season_standard_stats"

df_soccerdata_fbref_player_season_standard_stats = reduce(
    lambda a, b: a.unionByName(b),
    [flatten_two_row_header(p) for p in soccerdata_fbref_player_season_standard_stats_paths]
)

dim_season = spark.read.format("delta").load("s3://football-data-raw-01/silver/dim_season")
dim_players = spark.read.format("delta").load("s3://football-data-raw-01/silver/dim_players")
dim_team = spark.read.format("delta").load("s3://football-data-raw-01/silver/dim_team")

fact_players_season_standard_stats = (
    df_soccerdata_fbref_player_season_standard_stats
    .withColumnRenamed("season","fbref_season_id")
    .drop("league","nation","age")
)

fact_players_season_standard_stats = (
    fact_players_season_standard_stats.alias("f")
    .join(
        dim_season.alias("s"),
        F.expr("s.fbref_season_id = f.fbref_season_id"),
        "inner"
    )
    .join(
        dim_players.alias("p"),
        F.expr("p.player_name = f.player AND p.player_born = f.born"),
        "inner"
    )
    .join(
        dim_team.alias("t"),
        F.expr("t.fbref_team_name = f.team"),
        "inner"
    )
    .select(
        F.col("f.*"),
        F.col("s.season_id"),
        F.col("p.player_id"),
        F.col("t.team_id")
    )
    .drop("player","born","fbref_season_id","team")
)

fact_players_season_standard_stats = sanitize_columns(fact_players_season_standard_stats)
fact_players_season_standard_stats.write.format("delta").mode("overwrite").save(fact_players_season_standard_stats_path)

display(spark.read.format("delta").load(fact_players_season_standard_stats_path))

## fact_players_season_shooting_stats

In [0]:
from pyspark.sql import functions as F
from functools import reduce
from datetime import date

soccerdata_fbref_player_season_shooting_stats_paths = [
    f"s3://football-data-raw-01/raw/{year}/soccerdata_fbref/players_season_shooting_stats.csv"
    for year in range(PROJECT_START_SEASON_YEAR, date.today().year + 1)
]
soccerdata_fbref_player_season_shooting_stats_paths = [p for p in soccerdata_fbref_player_season_shooting_stats_paths if path_exists(p)]
fact_players_season_shooting_stats_path = "s3://football-data-raw-01/silver/fact_players_season_shooting_stats"

df_soccerdata_fbref_player_season_shooting_stats = reduce(
    lambda a, b: a.unionByName(b),
    [flatten_two_row_header(p) for p in soccerdata_fbref_player_season_shooting_stats_paths]
)

dim_season = spark.read.format("delta").load("s3://football-data-raw-01/silver/dim_season")
dim_players = spark.read.format("delta").load("s3://football-data-raw-01/silver/dim_players")
dim_team = spark.read.format("delta").load("s3://football-data-raw-01/silver/dim_team")

fact_players_season_shooting_stats = (
    df_soccerdata_fbref_player_season_shooting_stats
    .withColumnRenamed("season","fbref_season_id")
    .drop("league","nation","age")
)

fact_players_season_shooting_stats = (
    fact_players_season_shooting_stats.alias("f")
    .join(
        dim_season.alias("s"),
        F.expr("s.fbref_season_id = f.fbref_season_id"),
        "inner"
    )
    .join(
        dim_players.alias("p"),
        F.expr("p.player_name = f.player AND p.player_born = f.born"),
        "inner"
    )
    .join(
        dim_team.alias("t"),
        F.expr("t.fbref_team_name = f.team"),
        "inner"
    )
    .select(
        F.col("f.*"),
        F.col("s.season_id"),
        F.col("p.player_id"),
        F.col("t.team_id")
    )
    .drop("player","born","fbref_season_id","team")
)

fact_players_season_shooting_stats = sanitize_columns(fact_players_season_shooting_stats)
fact_players_season_shooting_stats.write.format("delta").mode("overwrite").save(fact_players_season_shooting_stats_path)

display(spark.read.format("delta").load(fact_players_season_shooting_stats_path))

## 

## fact_players_season_playing_time_stats

In [0]:
from pyspark.sql import functions as F
from functools import reduce
from datetime import date

soccerdata_fbref_player_season_playing_time_stats_paths = [
    f"s3://football-data-raw-01/raw/{year}/soccerdata_fbref/players_season_playing_time_stats.csv"
    for year in range(PROJECT_START_SEASON_YEAR, date.today().year + 1)
]
soccerdata_fbref_player_season_playing_time_stats_paths = [p for p in soccerdata_fbref_player_season_playing_time_stats_paths if path_exists(p)]
fact_players_season_playing_time_stats_path = "s3://football-data-raw-01/silver/fact_players_season_playing_time_stats"

df_soccerdata_fbref_player_season_playing_time_stats = reduce(
    lambda a, b: a.unionByName(b),
    [flatten_two_row_header(p) for p in soccerdata_fbref_player_season_playing_time_stats_paths]
)

dim_season = spark.read.format("delta").load("s3://football-data-raw-01/silver/dim_season")
dim_players = spark.read.format("delta").load("s3://football-data-raw-01/silver/dim_players")
dim_team = spark.read.format("delta").load("s3://football-data-raw-01/silver/dim_team")

fact_players_season_playing_time_stats = (
    df_soccerdata_fbref_player_season_playing_time_stats
    .withColumnRenamed("season","fbref_season_id")
    .drop("league","nation","age")
)

fact_players_season_playing_time_stats = (
    fact_players_season_playing_time_stats.alias("f")
    .join(
        dim_season.alias("s"),
        F.expr("s.fbref_season_id = f.fbref_season_id"),
        "inner"
    )
    .join(
        dim_players.alias("p"),
        F.expr("p.player_name = f.player AND p.player_born = f.born"),
        "inner"
    )
    .join(
        dim_team.alias("t"),
        F.expr("t.fbref_team_name = f.team"),
        "inner"
    )
    .select(
        F.col("f.*"),
        F.col("s.season_id"),
        F.col("p.player_id"),
        F.col("t.team_id")
    )
    .drop("player","born","fbref_season_id","team")
)

fact_players_season_playing_time_stats = sanitize_columns(fact_players_season_playing_time_stats)
fact_players_season_playing_time_stats.write.format("delta").mode("overwrite").save(fact_players_season_playing_time_stats_path)

display(spark.read.format("delta").load(fact_players_season_playing_time_stats_path))

## fact_players_season_misc_stats

In [0]:
from pyspark.sql import functions as F
from functools import reduce
from datetime import date

soccerdata_fbref_player_season_misc_stats_paths = [
    f"s3://football-data-raw-01/raw/{year}/soccerdata_fbref/players_season_misc_stats.csv"
    for year in range(PROJECT_START_SEASON_YEAR, date.today().year + 1)
]
soccerdata_fbref_player_season_misc_stats_paths = [p for p in soccerdata_fbref_player_season_misc_stats_paths if path_exists(p)]
fact_players_season_misc_stats_path = "s3://football-data-raw-01/silver/fact_players_season_misc_stats"

df_soccerdata_fbref_player_season_misc_stats = reduce(
    lambda a, b: a.unionByName(b),
    [flatten_two_row_header(p) for p in soccerdata_fbref_player_season_misc_stats_paths]
)

dim_season = spark.read.format("delta").load("s3://football-data-raw-01/silver/dim_season")
dim_players = spark.read.format("delta").load("s3://football-data-raw-01/silver/dim_players")
dim_team = spark.read.format("delta").load("s3://football-data-raw-01/silver/dim_team")

fact_players_season_misc_stats = (
    df_soccerdata_fbref_player_season_misc_stats
    .withColumnRenamed("season","fbref_season_id")
    .drop("league","nation","age","Performance_PKwon","Performance_PKcon")
)

fact_players_season_misc_stats = (
    fact_players_season_misc_stats.alias("f")
    .join(
        dim_season.alias("s"),
        F.expr("s.fbref_season_id = f.fbref_season_id"),
        "inner"
    )
    .join(
        dim_players.alias("p"),
        F.expr("p.player_name = f.player AND p.player_born = f.born"),
        "inner"
    )
    .join(
        dim_team.alias("t"),
        F.expr("t.fbref_team_name = f.team"),
        "inner"
    )
    .select(
        F.col("f.*"),
        F.col("s.season_id"),
        F.col("p.player_id"),
        F.col("t.team_id")
    )
    .drop("player","born","fbref_season_id","team")
)

fact_players_season_misc_stats = sanitize_columns(fact_players_season_misc_stats)
fact_players_season_misc_stats.write.format("delta").mode("overwrite").save(fact_players_season_misc_stats_path)

display(spark.read.format("delta").load(fact_players_season_misc_stats_path))

## fact_team_season_keeper_stats

In [0]:
from pyspark.sql import functions as F
from functools import reduce
from datetime import date

soccerdata_fbref_player_season_keeper_stats_paths = [
    f"s3://football-data-raw-01/raw/{year}/soccerdata_fbref/players_season_keeper_stats.csv"
    for year in range(PROJECT_START_SEASON_YEAR, date.today().year + 1)
]
soccerdata_fbref_player_season_keeper_stats_paths = [p for p in soccerdata_fbref_player_season_keeper_stats_paths if path_exists(p)]
fact_players_season_keeper_stats_path = "s3://football-data-raw-01/silver/fact_players_season_keeper_stats"

df_soccerdata_fbref_player_season_keeper_stats = reduce(
    lambda a, b: a.unionByName(b),
    [flatten_two_row_header(p) for p in soccerdata_fbref_player_season_keeper_stats_paths]
)

dim_season = spark.read.format("delta").load("s3://football-data-raw-01/silver/dim_season")
dim_players = spark.read.format("delta").load("s3://football-data-raw-01/silver/dim_players")
dim_team = spark.read.format("delta").load("s3://football-data-raw-01/silver/dim_team")

fact_players_season_keeper_stats = (
    df_soccerdata_fbref_player_season_keeper_stats
    .withColumnRenamed("season","fbref_season_id")
    .drop("league","nation","age")
)

fact_players_season_keeper_stats = (
    fact_players_season_keeper_stats.alias("f")
    .join(
        dim_season.alias("s"),
        F.expr("s.fbref_season_id = f.fbref_season_id"),
        "inner"
    )
    .join(
        dim_players.alias("p"),
        F.expr("p.player_name = f.player AND p.player_born = f.born"),
        "inner"
    )
    .join(
        dim_team.alias("t"),
        F.expr("t.fbref_team_name = f.team"),
        "inner"
    )
    .select(
        F.col("f.*"),
        F.col("s.season_id"),
        F.col("p.player_id"),
        F.col("t.team_id")
    )
    .drop("player","born","fbref_season_id","team")
)

fact_players_season_keeper_stats = sanitize_columns(fact_players_season_keeper_stats)
fact_players_season_keeper_stats.write.format("delta").mode("overwrite").save(fact_players_season_keeper_stats_path)

display(spark.read.format("delta").load(fact_players_season_keeper_stats_path))

# Gold

## fact_match_summary

In [0]:
from pyspark.sql import functions as F

dim_team_path = "s3://football-data-raw-01/silver/dim_team"
dim_team = spark.read.format("delta").load(dim_team_path)

fact_match_path = "s3://football-data-raw-01/silver/fact_match"
fact_match = spark.read.format("delta").load(fact_match_path)

dim_season_path = "s3://football-data-raw-01/silver/dim_season"
dim_season = spark.read.format("delta").load(dim_season_path)

fact_match_summary_gold_path = "s3://football-data-raw-01/gold/fact_match_summary"

fact_match_summary = (
    fact_match.alias("m")
    .join(
        dim_team.alias("t"),
        F.expr("t.team_id = m.home_team_id"),
        "inner"
    )
    .join(
        dim_team.alias("t2"),
        F.expr("t2.team_id = m.away_team_id"),
        "inner"
    )
    .join(
        dim_season.alias("s"),
        F.expr("s.season_id = m.season_id"),
        "inner"
    )
    .select(
        F.col("m.season_id"),
        F.col("s.season_name"),
        F.col("m.fact_match_id").alias("match_id"),
        F.col("gameweek"),
        F.col("m.match_date"),

        F.col("home_team_id"),
        F.col("t.team_tla").alias("home_team_tla"),
        F.col("t.team_name").alias("home_team_name"),
        F.col("t.team_crest").alias("home_team_crest"),
        F.col("m.half_time_home_team_score").alias("half_time_home_team_score"),
        F.col("m.full_time_home_team_score").alias("full_time_home_team_score"),
        F.col("m.home_team_red_cards"),

        F.col("away_team_id"),
        F.col("t2.team_tla").alias("away_team_tla"),
        F.col("t2.team_name").alias("away_team_name"),
        F.col("t2.team_crest").alias("away_team_crest"),
        F.col("m.half_time_away_team_score").alias("half_time_away_team_score"),
        F.col("m.full_time_away_team_score").alias("full_time_away_team_score"),
        F.col("m.away_team_red_cards"),

        F.col("m.first_half_injury_time"),
        F.col("m.second_half_injury_time"),
        F.col("t.team_venue").alias("stadium"),
        F.col("m.winner"),
        F.regexp_replace(F.col("m.match_referee"), r"[\[\]]", "").alias("match_referee"),

    )
)

fact_match_summary.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(fact_match_summary_gold_path)

display(fact_match_summary)

## fact_goals,fact_cards,fact_subs

In [0]:
from pyspark.sql import functions as F

fact_match_events_path = "s3://football-data-raw-01/silver/fact_match_events"
fact_match_events = spark.read.format("delta").load(fact_match_events_path)

dim_players_path = "s3://football-data-raw-01/silver/dim_players"
dim_players = spark.read.format("delta").load(dim_players_path)

fact_match_events = (
    fact_match_events.alias("e")
    .join(
        dim_players.alias("p1"),
        F.expr("e.player_1_id = p1.player_id"),
        "left"
    )
    .join(
        dim_players.alias("p2"),
        F.expr("e.player_2_id = p2.player_id"),
        "left"
    )
    .select(
        F.col("e.*"),
        F.col("p1.player_name").alias("player_1_name"),
        F.col("p2.player_name").alias("player_2_name")
    )
)

fact_goals = (
    fact_match_events
    .filter(F.col("event_type").isin("goal", "penalty_goal", "own_goal","penalty_miss"))
    .withColumn("is_penalty", F.col("event_type") == "penalty_goal")
    .withColumn("is_own_goal", F.col("event_type") == "own_goal")
    .withColumn("is_penalty_miss", F.col("event_type") == "penalty_miss")
    .withColumnRenamed("player_1_name", "scorer")
    .withColumnRenamed("player_2_name", "assist")
    .withColumnRenamed("player_1_id", "scorer_id")
    .withColumnRenamed("player_2_id", "assist_id")
    .drop("event_type")
)

fact_card = (
    fact_match_events
    .filter(F.col("event_type").isin("yellow_card", "red_card", "yellow_red_card"))
    .withColumn("yellow", F.col("event_type") == "yellow_card")
    .withColumn("red", F.col("event_type") == "red_card")
    .withColumn("second_yellow", F.col("event_type") == "yellow_red_card")
    .drop("event_type","player_2_id","player_2_name")
)

fact_subs = (
    fact_match_events
    .filter(F.col("event_type") == "substitute_in")
    .withColumnRenamed("player_1_name", "sub_in")
    .withColumnRenamed("player_1_id", "sub_in_id")
    .withColumnRenamed("player_2_name", "sub_out")
    .withColumnRenamed("player_2_id", "sub_out_id")
    .drop("event_type")
)

fact_goals_gold_path = "s3://football-data-raw-01/gold/fact_goals"
fact_card_gold_path = "s3://football-data-raw-01/gold/fact_card"
fact_subs_gold_path = "s3://football-data-raw-01/gold/fact_subs"

fact_goals.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(fact_goals_gold_path)
fact_card.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(fact_card_gold_path)
fact_subs.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(fact_subs_gold_path)



## fact_line_up_summary

In [0]:
fact_match_lineups_path = "s3://football-data-raw-01/silver/fact_match_lineups"
fact_match_lineups = spark.read.format("delta").load(fact_match_lineups_path)

fact_team_match_standard_stats_path = "s3://football-data-raw-01/silver/fact_team_match_standard_stats"
fact_team_match_standard_stats = spark.read.format("delta").load(fact_team_match_standard_stats_path)


fact_match_lineups = (
    fact_match_lineups.alias("l")
    # Only drop rows that can't be placed. Unused subs have no position and
    # 0 minutes - a blanket dropna() removed them (and anyone without a player_id).
    .dropna(subset=["match_id", "team_id", "player_id"])
    .join(
        fact_team_match_standard_stats.alias("t"), 
        F.expr("t.team_id = l.team_id AND t.season_id = l.season_id AND t.match_id = l.match_id" ),
        "inner"
    )
    .select(
        F.col("l.*"),
        F.col("t.Formation").alias("team_formation")
    )
)

fact_match_lineups_path = "s3://football-data-raw-01/gold/fact_match_lineups"
fact_match_lineups.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(fact_match_lineups_path)


## fact_team_season_performance

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType

fact_team_season_standard_stats_path = "s3://football-data-raw-01/silver/fact_team_season_standard_stats"
fact_team_season_standard_stats = spark.read.format("delta").load(fact_team_season_standard_stats_path)

fact_team_season_standard_stats = (
    fact_team_season_standard_stats
    .drop("Playing_Time_MP","Playing_Time_Starts","Playing_Time_Min","Playing_Time_90s")
    .withColumnRenamed("players_used", "standard_players_used")
    .withColumnRenamed("Age", "standard_avg_age")
    .withColumnRenamed("Poss", "standard_possession_pct")
    .withColumnRenamed("Performance_Gls", "standard_goals")
    .withColumnRenamed("Performance_Ast", "standard_assists")
    .withColumnRenamed("Performance_G+A", "standard_goals_plus_assists")
    .withColumnRenamed("Performance_G-PK", "standard_non_penalty_goals")
    .withColumnRenamed("Performance_PK", "standard_penalty_goals")
    .withColumnRenamed("Performance_PKatt", "standard_penalty_attempts")
    .withColumnRenamed("Performance_CrdY", "standard_yellow_cards")
    .withColumnRenamed("Performance_CrdR", "standard_red_cards")
    .withColumnRenamed("Per_90_Minutes_Gls", "standard_goals_per90")
    .withColumnRenamed("Per_90_Minutes_Ast", "standard_assists_per90")
    .withColumnRenamed("Per_90_Minutes_G+A", "standard_goals_plus_assists_per90")
    .withColumnRenamed("Per_90_Minutes_G-PK", "standard_non_penalty_goals_per90")
    .withColumnRenamed("Per_90_Minutes_G+A-PK", "standard_non_penalty_goals_plus_assists_per90")
    .withColumn("standard_players_used", F.coalesce(F.col("standard_players_used").cast(IntegerType()), F.lit(0)))
    .withColumn("standard_avg_age", F.col("standard_avg_age").cast(DoubleType()))
    .withColumn("standard_possession_pct", F.col("standard_possession_pct").cast(DoubleType()))
    .withColumn("standard_goals", F.coalesce(F.col("standard_goals").cast(IntegerType()), F.lit(0)))
    .withColumn("standard_assists", F.coalesce(F.col("standard_assists").cast(IntegerType()), F.lit(0)))
    .withColumn("standard_goals_plus_assists", F.coalesce(F.col("standard_goals_plus_assists").cast(IntegerType()), F.lit(0)))
    .withColumn("standard_non_penalty_goals", F.coalesce(F.col("standard_non_penalty_goals").cast(IntegerType()), F.lit(0)))
    .withColumn("standard_penalty_goals", F.coalesce(F.col("standard_penalty_goals").cast(IntegerType()), F.lit(0)))
    .withColumn("standard_penalty_attempts", F.coalesce(F.col("standard_penalty_attempts").cast(IntegerType()), F.lit(0)))
    .withColumn("standard_yellow_cards", F.coalesce(F.col("standard_yellow_cards").cast(IntegerType()), F.lit(0)))
    .withColumn("standard_red_cards", F.coalesce(F.col("standard_red_cards").cast(IntegerType()), F.lit(0)))
    .withColumn("standard_goals_per90", F.col("standard_goals_per90").cast(DoubleType()))
    .withColumn("standard_assists_per90", F.col("standard_assists_per90").cast(DoubleType()))
    .withColumn("standard_goals_plus_assists_per90", F.col("standard_goals_plus_assists_per90").cast(DoubleType()))
    .withColumn("standard_non_penalty_goals_per90", F.col("standard_non_penalty_goals_per90").cast(DoubleType()))
    .withColumn("standard_non_penalty_goals_plus_assists_per90", F.col("standard_non_penalty_goals_plus_assists_per90").cast(DoubleType()))
)

fact_team_season_shooting_stats_path = "s3://football-data-raw-01/silver/fact_team_season_shooting_stats"
fact_team_season_shooting_stats = spark.read.format("delta").load(fact_team_season_shooting_stats_path)

fact_team_season_shooting_stats = (
    fact_team_season_shooting_stats
    .drop("90s")
    .withColumnRenamed("players_used", "shooting_players_used")
    .withColumnRenamed("Standard_Gls", "shooting_goals")
    .withColumnRenamed("Standard_Sh", "shooting_shots")
    .withColumnRenamed("Standard_SoT", "shooting_shots_on_target")
    .withColumnRenamed("Standard_SoT%", "shooting_shots_on_target_pct")
    .withColumnRenamed("Standard_Sh/90", "shooting_shots_per90")
    .withColumnRenamed("Standard_SoT/90", "shooting_shots_on_target_per90")
    .withColumnRenamed("Standard_G/Sh", "shooting_goals_per_shot")
    .withColumnRenamed("Standard_G/SoT", "shooting_goals_per_shot_on_target")
    .withColumnRenamed("Standard_PK", "shooting_penalty_goals")
    .withColumnRenamed("Standard_PKatt", "shooting_penalty_attempts")
    .withColumn("shooting_players_used", F.coalesce(F.col("shooting_players_used").cast(IntegerType()), F.lit(0)))
    .withColumn("shooting_goals", F.coalesce(F.col("shooting_goals").cast(IntegerType()), F.lit(0)))
    .withColumn("shooting_shots", F.coalesce(F.col("shooting_shots").cast(IntegerType()), F.lit(0)))
    .withColumn("shooting_shots_on_target", F.coalesce(F.col("shooting_shots_on_target").cast(IntegerType()), F.lit(0)))
    .withColumn("shooting_shots_on_target_pct", F.col("shooting_shots_on_target_pct").cast(DoubleType()))
    .withColumn("shooting_shots_per90", F.col("shooting_shots_per90").cast(DoubleType()))
    .withColumn("shooting_shots_on_target_per90", F.col("shooting_shots_on_target_per90").cast(DoubleType()))
    .withColumn("shooting_goals_per_shot", F.col("shooting_goals_per_shot").cast(DoubleType()))
    .withColumn("shooting_goals_per_shot_on_target", F.col("shooting_goals_per_shot_on_target").cast(DoubleType()))
    .withColumn("shooting_penalty_goals", F.coalesce(F.col("shooting_penalty_goals").cast(IntegerType()), F.lit(0)))
    .withColumn("shooting_penalty_attempts", F.coalesce(F.col("shooting_penalty_attempts").cast(IntegerType()), F.lit(0)))
)

fact_team_season_playing_time_stats_path = "s3://football-data-raw-01/silver/fact_team_season_playing_time_stats"
fact_team_season_playing_time_stats = spark.read.format("delta").load(fact_team_season_playing_time_stats_path)

fact_team_season_playing_time_stats = (
    fact_team_season_playing_time_stats
    .drop("Age","players_used","Playing_Time_MP","Playing_Time_Min","Playing_Time_Mn/MP","Playing_Time_Min%","Playing_Time_90s","Starts_Starts")
    .withColumnRenamed("Starts_Mn/Start", "pt_avg_minutes_per_start")
    .withColumnRenamed("Starts_Compl", "pt_complete_matches")
    .withColumnRenamed("Subs_Subs", "pt_sub_appearances")
    .withColumnRenamed("Subs_Mn/Sub", "pt_avg_minutes_per_sub")
    .withColumnRenamed("Subs_unSub", "pt_unused_sub_appearances")
    .withColumnRenamed("Team_Success_PPM", "pt_points_per_match")
    .withColumnRenamed("Team_Success_onG", "pt_goals_scored_on_pitch")
    .withColumnRenamed("Team_Success_onGA", "pt_goals_conceded_on_pitch")
    .withColumnRenamed("Team_Success_+/-", "pt_plus_minus")
    .withColumnRenamed("Team_Success_+/-90", "pt_plus_minus_per90")
    .withColumn("pt_avg_minutes_per_start", F.col("pt_avg_minutes_per_start").cast(DoubleType()))
    .withColumn("pt_complete_matches", F.coalesce(F.col("pt_complete_matches").cast(IntegerType()), F.lit(0)))
    .withColumn("pt_sub_appearances", F.coalesce(F.col("pt_sub_appearances").cast(IntegerType()), F.lit(0)))
    .withColumn("pt_avg_minutes_per_sub", F.col("pt_avg_minutes_per_sub").cast(DoubleType()))
    .withColumn("pt_unused_sub_appearances", F.coalesce(F.col("pt_unused_sub_appearances").cast(IntegerType()), F.lit(0)))
    .withColumn("pt_points_per_match", F.col("pt_points_per_match").cast(DoubleType()))
    .withColumn("pt_goals_scored_on_pitch", F.coalesce(F.col("pt_goals_scored_on_pitch").cast(IntegerType()), F.lit(0)))
    .withColumn("pt_goals_conceded_on_pitch", F.coalesce(F.col("pt_goals_conceded_on_pitch").cast(IntegerType()), F.lit(0)))
    .withColumn("pt_plus_minus", F.coalesce(F.regexp_replace(F.col("pt_plus_minus"), r"\+", "").cast(IntegerType()), F.lit(0)))
    .withColumn("pt_plus_minus_per90", F.regexp_replace(F.col("pt_plus_minus_per90"), r"\+", "").cast(DoubleType()))
)

fact_team_season_misc_stats_path = "s3://football-data-raw-01/silver/fact_team_season_misc_stats"
fact_team_season_misc_stats = spark.read.format("delta").load(fact_team_season_misc_stats_path)

fact_team_season_misc_stats = (
    fact_team_season_misc_stats
    .drop("players_used","90s","Performance_CrdY","Performance_CrdR","Performance_PKcon")
    .withColumnRenamed("Performance_2CrdY", "misc_second_yellow_cards")
    .withColumnRenamed("Performance_Fls", "misc_fouls_committed")
    .withColumnRenamed("Performance_Fld", "misc_fouls_drawn")
    .withColumnRenamed("Performance_Off", "misc_offsides")
    .withColumnRenamed("Performance_Crs", "misc_crosses")
    .withColumnRenamed("Performance_Int", "misc_interceptions")
    .withColumnRenamed("Performance_TklW", "misc_tackles_won")
    .withColumnRenamed("Performance_OG", "misc_own_goals")
    .withColumn("misc_second_yellow_cards", F.coalesce(F.col("misc_second_yellow_cards").cast(IntegerType()), F.lit(0)))
    .withColumn("misc_fouls_committed", F.coalesce(F.col("misc_fouls_committed").cast(IntegerType()), F.lit(0)))
    .withColumn("misc_fouls_drawn", F.coalesce(F.col("misc_fouls_drawn").cast(IntegerType()), F.lit(0)))
    .withColumn("misc_offsides", F.coalesce(F.col("misc_offsides").cast(IntegerType()), F.lit(0)))
    .withColumn("misc_crosses", F.coalesce(F.col("misc_crosses").cast(IntegerType()), F.lit(0)))
    .withColumn("misc_interceptions", F.coalesce(F.col("misc_interceptions").cast(IntegerType()), F.lit(0)))
    .withColumn("misc_tackles_won", F.coalesce(F.col("misc_tackles_won").cast(IntegerType()), F.lit(0)))
    .withColumn("misc_own_goals", F.coalesce(F.col("misc_own_goals").cast(IntegerType()), F.lit(0)))
)

fact_team_season_keeper_stats_path = "s3://football-data-raw-01/silver/fact_team_season_keeper_stats"
fact_team_season_keeper_stats = spark.read.format("delta").load(fact_team_season_keeper_stats_path)

fact_team_season_keeper_stats = (
    fact_team_season_keeper_stats
    .drop("Playing_Time_MP","Playing_Time_Min","Playing_Time_Starts","Playing_Time_90s","Performance_GA","Performance_GA90")
    .withColumnRenamed("players_used", "gk_players_used")
    .withColumnRenamed("Performance_SoTA", "gk_shots_on_target_against")
    .withColumnRenamed("Performance_Saves", "gk_saves")
    .withColumnRenamed("Performance_Save%", "gk_save_pct")
    .withColumnRenamed("Performance_W", "gk_wins")
    .withColumnRenamed("Performance_D", "gk_draws")
    .withColumnRenamed("Performance_L", "gk_losses")
    .withColumnRenamed("Performance_CS", "gk_clean_sheets")
    .withColumnRenamed("Performance_CS%", "gk_clean_sheet_pct")
    .withColumnRenamed("Penalty_Kicks_PKatt", "gk_penalty_kicks_faced")
    .withColumnRenamed("Penalty_Kicks_PKA", "gk_penalty_kicks_allowed")
    .withColumnRenamed("Penalty_Kicks_PKsv", "gk_penalty_kicks_saved")
    .withColumnRenamed("Penalty_Kicks_PKm", "gk_penalty_kicks_missed")
    .withColumnRenamed("Penalty_Kicks_Save%", "gk_penalty_save_pct")
    .withColumn("gk_players_used", F.coalesce(F.col("gk_players_used").cast(IntegerType()), F.lit(0)))
    .withColumn("gk_shots_on_target_against", F.coalesce(F.col("gk_shots_on_target_against").cast(IntegerType()), F.lit(0)))
    .withColumn("gk_saves", F.coalesce(F.col("gk_saves").cast(IntegerType()), F.lit(0)))
    .withColumn("gk_save_pct", F.col("gk_save_pct").cast(DoubleType()))
    .withColumn("gk_wins", F.coalesce(F.col("gk_wins").cast(IntegerType()), F.lit(0)))
    .withColumn("gk_draws", F.coalesce(F.col("gk_draws").cast(IntegerType()), F.lit(0)))
    .withColumn("gk_losses", F.coalesce(F.col("gk_losses").cast(IntegerType()), F.lit(0)))
    .withColumn("gk_clean_sheets", F.coalesce(F.col("gk_clean_sheets").cast(IntegerType()), F.lit(0)))
    .withColumn("gk_clean_sheet_pct", F.col("gk_clean_sheet_pct").cast(DoubleType()))
    .withColumn("gk_penalty_kicks_faced", F.coalesce(F.col("gk_penalty_kicks_faced").cast(IntegerType()), F.lit(0)))
    .withColumn("gk_penalty_kicks_allowed", F.coalesce(F.col("gk_penalty_kicks_allowed").cast(IntegerType()), F.lit(0)))
    .withColumn("gk_penalty_kicks_saved", F.coalesce(F.col("gk_penalty_kicks_saved").cast(IntegerType()), F.lit(0)))
    .withColumn("gk_penalty_kicks_missed", F.coalesce(F.col("gk_penalty_kicks_missed").cast(IntegerType()), F.lit(0)))
    .withColumn("gk_penalty_save_pct", F.col("gk_penalty_save_pct").cast(DoubleType()))
)

fact_team_season_stats = (
    fact_team_season_standard_stats
    .join(fact_team_season_keeper_stats, on = ["season_id", "team_id"], how = "inner")
    .join(fact_team_season_shooting_stats, on = ["season_id", "team_id"], how = "inner")
    .join(fact_team_season_misc_stats, on = ["season_id", "team_id"], how = "inner")
    .join(fact_team_season_playing_time_stats, on = ["season_id", "team_id"], how = "inner")
)

fact_team_season_stats_path = "s3://football-data-raw-01/gold/fact_team_season_stats"

fact_team_season_stats.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(fact_team_season_stats_path)


## fact_player_season_stats


In [0]:

from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType

fact_players_season_standard_stats_path = "s3://football-data-raw-01/silver/fact_players_season_standard_stats"
fact_players_season_standard_stats = spark.read.format("delta").load(fact_players_season_standard_stats_path)

fact_players_season_standard_stats = (
    fact_players_season_standard_stats
    .withColumnRenamed("pos", "position")
    .withColumnRenamed("Playing_Time_MP", "standard_matches_played")
    .withColumnRenamed("Playing_Time_Starts", "standard_starts")
    .withColumnRenamed("Playing_Time_Min", "standard_minutes_played")
    .withColumnRenamed("Playing_Time_90s", "standard_nineties_played")
    .withColumnRenamed("Performance_Gls", "standard_goals")
    .withColumnRenamed("Performance_Ast", "standard_assists")
    .withColumnRenamed("Performance_G+A", "standard_goals_plus_assists")
    .withColumnRenamed("Performance_G-PK", "standard_non_penalty_goals")
    .withColumnRenamed("Performance_PK", "standard_penalty_goals")
    .withColumnRenamed("Performance_PKatt", "standard_penalty_attempts")
    .withColumnRenamed("Performance_CrdY", "standard_yellow_cards")
    .withColumnRenamed("Performance_CrdR", "standard_red_cards")
    .withColumnRenamed("Per_90_Minutes_Gls", "standard_goals_per90")
    .withColumnRenamed("Per_90_Minutes_Ast", "standard_assists_per90")
    .withColumnRenamed("Per_90_Minutes_G+A", "standard_goals_plus_assists_per90")
    .withColumnRenamed("Per_90_Minutes_G-PK", "standard_non_penalty_goals_per90")
    .withColumnRenamed("Per_90_Minutes_G+A-PK", "standard_non_penalty_goals_plus_assists_per90")
    .withColumn("standard_matches_played", F.coalesce(F.col("standard_matches_played").cast(IntegerType()), F.lit(0)))
    .withColumn("standard_starts", F.coalesce(F.col("standard_starts").cast(IntegerType()), F.lit(0)))
    .withColumn("standard_minutes_played", F.coalesce(F.col("standard_minutes_played").cast(IntegerType()), F.lit(0)))
    .withColumn("standard_nineties_played", F.col("standard_nineties_played").cast(DoubleType()))
    .withColumn("standard_goals", F.coalesce(F.col("standard_goals").cast(IntegerType()), F.lit(0)))
    .withColumn("standard_assists", F.coalesce(F.col("standard_assists").cast(IntegerType()), F.lit(0)))
    .withColumn("standard_goals_plus_assists", F.coalesce(F.col("standard_goals_plus_assists").cast(IntegerType()), F.lit(0)))
    .withColumn("standard_non_penalty_goals", F.coalesce(F.col("standard_non_penalty_goals").cast(IntegerType()), F.lit(0)))
    .withColumn("standard_penalty_goals", F.coalesce(F.col("standard_penalty_goals").cast(IntegerType()), F.lit(0)))
    .withColumn("standard_penalty_attempts", F.coalesce(F.col("standard_penalty_attempts").cast(IntegerType()), F.lit(0)))
    .withColumn("standard_yellow_cards", F.coalesce(F.col("standard_yellow_cards").cast(IntegerType()), F.lit(0)))
    .withColumn("standard_red_cards", F.coalesce(F.col("standard_red_cards").cast(IntegerType()), F.lit(0)))
    .withColumn("standard_goals_per90", F.col("standard_goals_per90").cast(DoubleType()))
    .withColumn("standard_assists_per90", F.col("standard_assists_per90").cast(DoubleType()))
    .withColumn("standard_goals_plus_assists_per90", F.col("standard_goals_plus_assists_per90").cast(DoubleType()))
    .withColumn("standard_non_penalty_goals_per90", F.col("standard_non_penalty_goals_per90").cast(DoubleType()))
    .withColumn("standard_non_penalty_goals_plus_assists_per90", F.col("standard_non_penalty_goals_plus_assists_per90").cast(DoubleType()))
)

fact_players_season_shooting_stats_path = "s3://football-data-raw-01/silver/fact_players_season_shooting_stats"
fact_players_season_shooting_stats = spark.read.format("delta").load(fact_players_season_shooting_stats_path)

fact_players_season_shooting_stats = (
    fact_players_season_shooting_stats
    .drop("pos", "90s", "Standard_Gls", "Standard_PK", "Standard_PKatt")
    .withColumnRenamed("Standard_Sh", "shooting_total_shots")
    .withColumnRenamed("Standard_SoT", "shooting_shots_on_target")
    .withColumnRenamed("Standard_SoT%", "shooting_shots_on_target_pct")
    .withColumnRenamed("Standard_Sh/90", "shooting_shots_per90")
    .withColumnRenamed("Standard_SoT/90", "shooting_shots_on_target_per90")
    .withColumnRenamed("Standard_G/Sh", "shooting_goals_per_shot")
    .withColumnRenamed("Standard_G/SoT", "shooting_goals_per_shot_on_target")
    .withColumn("shooting_total_shots", F.coalesce(F.col("shooting_total_shots").cast(IntegerType()), F.lit(0)))
    .withColumn("shooting_shots_on_target", F.coalesce(F.col("shooting_shots_on_target").cast(IntegerType()), F.lit(0)))
    .withColumn("shooting_shots_on_target_pct", F.col("shooting_shots_on_target_pct").cast(DoubleType()))
    .withColumn("shooting_shots_per90", F.col("shooting_shots_per90").cast(DoubleType()))
    .withColumn("shooting_shots_on_target_per90", F.col("shooting_shots_on_target_per90").cast(DoubleType()))
    .withColumn("shooting_goals_per_shot", F.col("shooting_goals_per_shot").cast(DoubleType()))
    .withColumn("shooting_goals_per_shot_on_target", F.col("shooting_goals_per_shot_on_target").cast(DoubleType()))
)

fact_players_season_playing_time_stats_path = "s3://football-data-raw-01/silver/fact_players_season_playing_time_stats"
fact_players_season_playing_time_stats = spark.read.format("delta").load(fact_players_season_playing_time_stats_path)

fact_players_season_playing_time_stats = (
    fact_players_season_playing_time_stats
    .drop("pos", "Playing_Time_MP", "Playing_Time_Min", "Playing_Time_90s", "Starts_Starts")
    .withColumnRenamed("Playing_Time_Mn/MP", "pt_avg_minutes_per_match")
    .withColumnRenamed("Playing_Time_Min%", "pt_minutes_pct")
    .withColumnRenamed("Starts_Mn/Start", "pt_avg_minutes_per_start")
    .withColumnRenamed("Starts_Compl", "pt_complete_matches")
    .withColumnRenamed("Subs_Subs", "pt_sub_appearances")
    .withColumnRenamed("Subs_Mn/Sub", "pt_avg_minutes_per_sub")
    .withColumnRenamed("Subs_unSub", "pt_unused_sub_appearances")
    .withColumnRenamed("Team_Success_PPM", "pt_points_per_match")
    .withColumnRenamed("Team_Success_onG", "pt_goals_scored_on_pitch")
    .withColumnRenamed("Team_Success_onGA", "pt_goals_conceded_on_pitch")
    .withColumnRenamed("Team_Success_+/-", "pt_plus_minus")
    .withColumnRenamed("Team_Success_+/-90", "pt_plus_minus_per90")
    .withColumnRenamed("Team_Success_On-Off", "pt_on_off")
    .withColumn("pt_avg_minutes_per_match", F.col("pt_avg_minutes_per_match").cast(DoubleType()))
    .withColumn("pt_minutes_pct", F.col("pt_minutes_pct").cast(DoubleType()))
    .withColumn("pt_avg_minutes_per_start", F.col("pt_avg_minutes_per_start").cast(DoubleType()))
    .withColumn("pt_complete_matches", F.coalesce(F.col("pt_complete_matches").cast(IntegerType()), F.lit(0)))
    .withColumn("pt_sub_appearances", F.coalesce(F.col("pt_sub_appearances").cast(IntegerType()), F.lit(0)))
    .withColumn("pt_avg_minutes_per_sub", F.col("pt_avg_minutes_per_sub").cast(DoubleType()))
    .withColumn("pt_unused_sub_appearances", F.coalesce(F.col("pt_unused_sub_appearances").cast(IntegerType()), F.lit(0)))
    .withColumn("pt_points_per_match", F.col("pt_points_per_match").cast(DoubleType()))
    .withColumn("pt_goals_scored_on_pitch", F.coalesce(F.col("pt_goals_scored_on_pitch").cast(IntegerType()), F.lit(0)))
    .withColumn("pt_goals_conceded_on_pitch", F.coalesce(F.col("pt_goals_conceded_on_pitch").cast(IntegerType()), F.lit(0)))
    .withColumn("pt_plus_minus", F.coalesce(F.regexp_replace(F.col("pt_plus_minus"), r"\+", "").cast(IntegerType()), F.lit(0)))
    .withColumn("pt_plus_minus_per90", F.regexp_replace(F.col("pt_plus_minus_per90"), r"\+", "").cast(DoubleType()))
    .withColumn("pt_on_off", F.regexp_replace(F.col("pt_on_off"), r"\+", "").cast(DoubleType()))
)

fact_players_season_misc_stats_path = "s3://football-data-raw-01/silver/fact_players_season_misc_stats"
fact_players_season_misc_stats = spark.read.format("delta").load(fact_players_season_misc_stats_path)
fact_players_season_misc_stats = (
    fact_players_season_misc_stats
    .drop("pos", "90s", "Performance_CrdY", "Performance_CrdR")
    .withColumnRenamed("Performance_2CrdY", "misc_second_yellow_cards")
    .withColumnRenamed("Performance_Fls", "misc_fouls_committed")
    .withColumnRenamed("Performance_Fld", "misc_fouls_drawn")
    .withColumnRenamed("Performance_Off", "misc_offsides")
    .withColumnRenamed("Performance_Crs", "misc_crosses")
    .withColumnRenamed("Performance_Int", "misc_interceptions")
    .withColumnRenamed("Performance_TklW", "misc_tackles_won")
    .withColumnRenamed("Performance_OG", "misc_own_goals")
    .withColumn("misc_second_yellow_cards", F.coalesce(F.col("misc_second_yellow_cards").cast(IntegerType()), F.lit(0)))
    .withColumn("misc_fouls_committed", F.coalesce(F.col("misc_fouls_committed").cast(IntegerType()), F.lit(0)))
    .withColumn("misc_fouls_drawn", F.coalesce(F.col("misc_fouls_drawn").cast(IntegerType()), F.lit(0)))
    .withColumn("misc_offsides", F.coalesce(F.col("misc_offsides").cast(IntegerType()), F.lit(0)))
    .withColumn("misc_crosses", F.coalesce(F.col("misc_crosses").cast(IntegerType()), F.lit(0)))
    .withColumn("misc_interceptions", F.coalesce(F.col("misc_interceptions").cast(IntegerType()), F.lit(0)))
    .withColumn("misc_tackles_won", F.coalesce(F.col("misc_tackles_won").cast(IntegerType()), F.lit(0)))
    .withColumn("misc_own_goals", F.coalesce(F.col("misc_own_goals").cast(IntegerType()), F.lit(0)))
)

fact_players_season_keeper_stats_path = "s3://football-data-raw-01/silver/fact_players_season_keeper_stats"
fact_players_season_keeper_stats = spark.read.format("delta").load(fact_players_season_keeper_stats_path)
fact_players_season_keeper_stats = (
    fact_players_season_keeper_stats
    .drop("pos", "Playing_Time_MP", "Playing_Time_Starts", "Playing_Time_Min", "Playing_Time_90s")
    .withColumnRenamed("Performance_GA", "gk_goals_against")
    .withColumnRenamed("Performance_GA90", "gk_goals_against_per90")
    .withColumnRenamed("Performance_SoTA", "gk_shots_on_target_against")
    .withColumnRenamed("Performance_Saves", "gk_saves")
    .withColumnRenamed("Performance_Save%", "gk_save_pct")
    .withColumnRenamed("Performance_W", "gk_wins")
    .withColumnRenamed("Performance_D", "gk_draws")
    .withColumnRenamed("Performance_L", "gk_losses")
    .withColumnRenamed("Performance_CS", "gk_clean_sheets")
    .withColumnRenamed("Performance_CS%", "gk_clean_sheet_pct")
    .withColumnRenamed("Penalty_Kicks_PKatt", "gk_penalty_kicks_faced")
    .withColumnRenamed("Penalty_Kicks_PKA", "gk_penalty_kicks_allowed")
    .withColumnRenamed("Penalty_Kicks_PKsv", "gk_penalty_kicks_saved")
    .withColumnRenamed("Penalty_Kicks_PKm", "gk_penalty_kicks_missed")
    .withColumnRenamed("Penalty_Kicks_Save%", "gk_penalty_save_pct")
    .withColumn("gk_goals_against", F.coalesce(F.col("gk_goals_against").cast(IntegerType()), F.lit(0)))
    .withColumn("gk_goals_against_per90", F.col("gk_goals_against_per90").cast(DoubleType()))
    .withColumn("gk_shots_on_target_against", F.coalesce(F.col("gk_shots_on_target_against").cast(IntegerType()), F.lit(0)))
    .withColumn("gk_saves", F.coalesce(F.col("gk_saves").cast(IntegerType()), F.lit(0)))
    .withColumn("gk_save_pct", F.col("gk_save_pct").cast(DoubleType()))
    .withColumn("gk_wins", F.coalesce(F.col("gk_wins").cast(IntegerType()), F.lit(0)))
    .withColumn("gk_draws", F.coalesce(F.col("gk_draws").cast(IntegerType()), F.lit(0)))
    .withColumn("gk_losses", F.coalesce(F.col("gk_losses").cast(IntegerType()), F.lit(0)))
    .withColumn("gk_clean_sheets", F.coalesce(F.col("gk_clean_sheets").cast(IntegerType()), F.lit(0)))
    .withColumn("gk_clean_sheet_pct", F.col("gk_clean_sheet_pct").cast(DoubleType()))
    .withColumn("gk_penalty_kicks_faced", F.coalesce(F.col("gk_penalty_kicks_faced").cast(IntegerType()), F.lit(0)))
    .withColumn("gk_penalty_kicks_allowed", F.coalesce(F.col("gk_penalty_kicks_allowed").cast(IntegerType()), F.lit(0)))
    .withColumn("gk_penalty_kicks_saved", F.coalesce(F.col("gk_penalty_kicks_saved").cast(IntegerType()), F.lit(0)))
    .withColumn("gk_penalty_kicks_missed", F.coalesce(F.col("gk_penalty_kicks_missed").cast(IntegerType()), F.lit(0)))
    .withColumn("gk_penalty_save_pct", F.col("gk_penalty_save_pct").cast(DoubleType()))
)

fact_players_season_stats = (
    fact_players_season_standard_stats
    .join(fact_players_season_shooting_stats, on=["player_id", "season_id","team_id"],how ="left")
    .join(fact_players_season_playing_time_stats, on=["player_id", "season_id","team_id"],how ="left")
    .join(fact_players_season_misc_stats, on=["player_id", "season_id","team_id"],how="left")
    .join(fact_players_season_keeper_stats, on=["player_id", "season_id","team_id"],how="left")
)

fact_players_season_stats_path = "s3://football-data-raw-01/gold/fact_players_season_stats"

fact_players_season_stats.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(fact_players_season_stats_path)

## fact_table

In [0]:
from pyspark.sql import functions as F
fact_standing_path = "s3://football-data-raw-01/silver/fact_standing"
fact_standing = spark.read.format("delta").load(fact_standing_path)

dim_team_path = "s3://football-data-raw-01/silver/dim_team"
dim_team = spark.read.format("delta").load(dim_team_path)


fact_table = (
    fact_standing.alias("s")
    .join(dim_team.alias("t"), on="team_id", how="inner")
    .select(
        F.col("s.*"),
        F.col("t.team_name"),
        F.col("t.team_crest"),
        F.col("t.team_tla")
    )
)

fact_table_path = "s3://football-data-raw-01/gold/fact_table"

fact_table.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(fact_table_path)

## fact_team_match_performance

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType

fact_team_match_standard_stats_path = "s3://football-data-raw-01/silver/fact_team_match_standard_stats"
fact_team_match_standard_stats = spark.read.format("delta").load(fact_team_match_standard_stats_path)
fact_team_match_standard_stats = (
    fact_team_match_standard_stats
    .withColumnRenamed("Formation", "team_formation")
    .withColumnRenamed("Opp_Formation", "opponent_formation")
    .withColumnRenamed("Poss", "possession_pct")
)

fact_team_match_shooting_stats_path = "s3://football-data-raw-01/silver/fact_team_match_shooting_stats"
fact_team_match_shooting_stats = spark.read.format("delta").load(fact_team_match_shooting_stats_path)
fact_team_match_shooting_stats = (
    fact_team_match_shooting_stats
    .withColumnRenamed("Standard_Gls", "shooting_goals")
    .withColumnRenamed("Standard_Sh", "shooting_shots")
    .withColumnRenamed("Standard_SoT", "shooting_shots_on_target")
    .withColumnRenamed("Standard_SoT%", "shooting_shots_on_target_pct")
    .withColumnRenamed("Standard_G/Sh", "shooting_goals_per_shot")
    .withColumnRenamed("Standard_G/SoT", "shooting_goals_per_shot_on_target")
    .withColumnRenamed("Standard_PK", "shooting_penalty_goals")
    .withColumnRenamed("Standard_PKatt", "shooting_penalty_attempts")
    .withColumn("shooting_goals", F.coalesce(F.col("shooting_goals").cast(IntegerType()), F.lit(0)))
    .withColumn("shooting_shots", F.coalesce(F.col("shooting_shots").cast(IntegerType()), F.lit(0)))
    .withColumn("shooting_shots_on_target", F.coalesce(F.col("shooting_shots_on_target").cast(IntegerType()), F.lit(0)))
    .withColumn("shooting_shots_on_target_pct", F.col("shooting_shots_on_target_pct").cast(DoubleType()))
    .withColumn("shooting_goals_per_shot", F.col("shooting_goals_per_shot").cast(DoubleType()))
    .withColumn("shooting_goals_per_shot_on_target", F.col("shooting_goals_per_shot_on_target").cast(DoubleType()))
    .withColumn("shooting_penalty_goals", F.coalesce(F.col("shooting_penalty_goals").cast(IntegerType()), F.lit(0)))
    .withColumn("shooting_penalty_attempts", F.coalesce(F.col("shooting_penalty_attempts").cast(IntegerType()), F.lit(0)))
)

fact_team_match_misc_stats_path = "s3://football-data-raw-01/silver/fact_team_match_misc_stats"
fact_team_match_misc_stats = spark.read.format("delta").load(fact_team_match_misc_stats_path)
fact_team_match_misc_stats = (
    fact_team_match_misc_stats
    .withColumnRenamed("Performance_CrdY", "misc_yellow_cards")
    .withColumnRenamed("Performance_CrdR", "misc_red_cards")
    .withColumnRenamed("Performance_2CrdY", "misc_second_yellow_cards")
    .withColumnRenamed("Performance_Fls", "misc_fouls_committed")
    .withColumnRenamed("Performance_Fld", "misc_fouls_drawn")
    .withColumnRenamed("Performance_Off", "misc_offsides")
    .withColumnRenamed("Performance_Crs", "misc_crosses")
    .withColumnRenamed("Performance_Int", "misc_interceptions")
    .withColumnRenamed("Performance_TklW", "misc_tackles_won")
    .withColumnRenamed("Performance_OG", "misc_own_goals")
    .withColumn("misc_yellow_cards", F.coalesce(F.col("misc_yellow_cards").cast(IntegerType()), F.lit(0)))
    .withColumn("misc_red_cards", F.coalesce(F.col("misc_red_cards").cast(IntegerType()), F.lit(0)))
    .withColumn("misc_second_yellow_cards", F.coalesce(F.col("misc_second_yellow_cards").cast(IntegerType()), F.lit(0)))
    .withColumn("misc_fouls_committed", F.coalesce(F.col("misc_fouls_committed").cast(IntegerType()), F.lit(0)))
    .withColumn("misc_fouls_drawn", F.coalesce(F.col("misc_fouls_drawn").cast(IntegerType()), F.lit(0)))
    .withColumn("misc_offsides", F.coalesce(F.col("misc_offsides").cast(IntegerType()), F.lit(0)))
    .withColumn("misc_crosses", F.coalesce(F.col("misc_crosses").cast(IntegerType()), F.lit(0)))
    .withColumn("misc_interceptions", F.coalesce(F.col("misc_interceptions").cast(IntegerType()), F.lit(0)))
    .withColumn("misc_tackles_won", F.coalesce(F.col("misc_tackles_won").cast(IntegerType()), F.lit(0)))
    .withColumn("misc_own_goals", F.coalesce(F.col("misc_own_goals").cast(IntegerType()), F.lit(0)))
)

fact_team_match_keeper_stats_path = "s3://football-data-raw-01/silver/fact_team_match_keeper_stats"
fact_team_match_keeper_stats = spark.read.format("delta").load(fact_team_match_keeper_stats_path)
fact_team_match_keeper_stats = (
    fact_team_match_keeper_stats
    .withColumnRenamed("Performance_SoTA", "gk_shots_on_target_against")
    .withColumnRenamed("Performance_GA", "gk_goals_against")
    .withColumnRenamed("Performance_Saves", "gk_saves")
    .withColumnRenamed("Performance_Save%", "gk_save_pct")
    .withColumnRenamed("Performance_CS", "gk_clean_sheet")
    .withColumnRenamed("Penalty_Kicks_PKatt", "gk_penalty_kicks_faced")
    .withColumnRenamed("Penalty_Kicks_PKA", "gk_penalty_kicks_allowed")
    .withColumnRenamed("Penalty_Kicks_PKsv", "gk_penalty_kicks_saved")
    .withColumnRenamed("Penalty_Kicks_PKm", "gk_penalty_kicks_missed")
    .withColumn("gk_shots_on_target_against", F.coalesce(F.col("gk_shots_on_target_against").cast(IntegerType()), F.lit(0)))
    .withColumn("gk_goals_against", F.coalesce(F.col("gk_goals_against").cast(IntegerType()), F.lit(0)))
    .withColumn("gk_saves", F.coalesce(F.col("gk_saves").cast(IntegerType()), F.lit(0)))
    .withColumn("gk_save_pct", F.col("gk_save_pct").cast(DoubleType()))
    .withColumn("gk_clean_sheet", F.coalesce(F.col("gk_clean_sheet").cast(IntegerType()), F.lit(0)))
    .withColumn("gk_penalty_kicks_faced", F.coalesce(F.col("gk_penalty_kicks_faced").cast(IntegerType()), F.lit(0)))
    .withColumn("gk_penalty_kicks_allowed", F.coalesce(F.col("gk_penalty_kicks_allowed").cast(IntegerType()), F.lit(0)))
    .withColumn("gk_penalty_kicks_saved", F.coalesce(F.col("gk_penalty_kicks_saved").cast(IntegerType()), F.lit(0)))
    .withColumn("gk_penalty_kicks_missed", F.coalesce(F.col("gk_penalty_kicks_missed").cast(IntegerType()), F.lit(0)))
)

dim_players_path = "s3://football-data-raw-01/silver/dim_players"
dim_players = spark.read.format("delta").load(dim_players_path)

fact_team_match_performance = (
    fact_team_match_standard_stats.alias("fs")
    .join(fact_team_match_shooting_stats.alias("fss"),on=["match_id","team_id","season_id"],how="inner")
    .join(fact_team_match_misc_stats.alias("fm"),on=["match_id","team_id","season_id"],how="inner")
    .join(fact_team_match_keeper_stats.alias("fk"),on=["match_id","team_id","season_id"],how="inner")
)

fact_team_match_performance = (
    fact_team_match_performance.alias("fs")
    .join(
        dim_players.alias("p"),
        F.expr("fs.captain_id = p.player_id"),
        "inner"
    )
    .select(
        F.col("fs.*"),
        F.col("p.player_name").alias("captain_name")

    )
)
fact_team_match_performance_path = "s3://football-data-raw-01/gold/fact_team_match_performance"
fact_team_match_performance.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(fact_team_match_performance_path)

display(fact_team_match_performance.printSchema)


## fact_player_match_performance,fact_goal_keeper_match_performance

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType

fact_player_match_stats_summary_path = "s3://football-data-raw-01/silver/fact_player_match_stats_summary"
fact_player_match_stats_summary = spark.read.format("delta").load(fact_player_match_stats_summary_path)
fact_player_match_stats_summary = (
    fact_player_match_stats_summary
    .withColumnRenamed("jersey_number", "jersey_number")
    .withColumnRenamed("pos", "position")
    .withColumnRenamed("min", "minutes_played")
    .withColumnRenamed("Performance_Gls", "goals")
    .withColumnRenamed("Performance_Ast", "assists")
    .withColumnRenamed("Performance_PK", "penalty_goals")
    .withColumnRenamed("Performance_PKatt", "penalty_attempts")
    .withColumnRenamed("Performance_Sh", "shots")
    .withColumnRenamed("Performance_SoT", "shots_on_target")
    .withColumnRenamed("Performance_CrdY", "yellow_cards")
    .withColumnRenamed("Performance_CrdR", "red_cards")
    .withColumnRenamed("Performance_Fls", "fouls_committed")
    .withColumnRenamed("Performance_Fld", "fouls_drawn")
    .withColumnRenamed("Performance_Off", "offsides")
    .withColumnRenamed("Performance_Crs", "crosses")
    .withColumnRenamed("Performance_TklW", "tackles_won")
    .withColumnRenamed("Performance_Int", "interceptions")
    .withColumnRenamed("Performance_OG", "own_goals")
    .withColumn("jersey_number", F.col("jersey_number").cast(IntegerType()))
    .withColumn("minutes_played", F.coalesce(F.col("minutes_played").cast(IntegerType()), F.lit(0)))
    .withColumn("goals", F.coalesce(F.col("goals").cast(IntegerType()), F.lit(0)))
    .withColumn("assists", F.coalesce(F.col("assists").cast(IntegerType()), F.lit(0)))
    .withColumn("penalty_goals", F.coalesce(F.col("penalty_goals").cast(IntegerType()), F.lit(0)))
    .withColumn("penalty_attempts", F.coalesce(F.col("penalty_attempts").cast(IntegerType()), F.lit(0)))
    .withColumn("shots", F.coalesce(F.col("shots").cast(IntegerType()), F.lit(0)))
    .withColumn("shots_on_target", F.coalesce(F.col("shots_on_target").cast(IntegerType()), F.lit(0)))
    .withColumn("yellow_cards", F.coalesce(F.col("yellow_cards").cast(IntegerType()), F.lit(0)))
    .withColumn("red_cards", F.coalesce(F.col("red_cards").cast(IntegerType()), F.lit(0)))
    .withColumn("fouls_committed", F.coalesce(F.col("fouls_committed").cast(IntegerType()), F.lit(0)))
    .withColumn("fouls_drawn", F.coalesce(F.col("fouls_drawn").cast(IntegerType()), F.lit(0)))
    .withColumn("offsides", F.coalesce(F.col("offsides").cast(IntegerType()), F.lit(0)))
    .withColumn("crosses", F.coalesce(F.col("crosses").cast(IntegerType()), F.lit(0)))
    .withColumn("tackles_won", F.coalesce(F.col("tackles_won").cast(IntegerType()), F.lit(0)))
    .withColumn("interceptions", F.coalesce(F.col("interceptions").cast(IntegerType()), F.lit(0)))
    .withColumn("own_goals", F.coalesce(F.col("own_goals").cast(IntegerType()), F.lit(0)))
)

fact_player_match_keepers_summary_path = "s3://football-data-raw-01/silver/fact_player_match_keepers_summary"
fact_player_match_keepers_summary = spark.read.format("delta").load(fact_player_match_keepers_summary_path)
fact_player_match_keepers_summary = (
    fact_player_match_keepers_summary
    .withColumnRenamed("min", "minutes_played")
    .withColumnRenamed("Shot_Stopping_SoTA", "gk_shots_on_target_against")
    .withColumnRenamed("Shot_Stopping_GA", "gk_goals_against")
    .withColumnRenamed("Shot_Stopping_Saves", "gk_saves")
    .withColumnRenamed("Shot_Stopping_Save%", "gk_save_pct")
    .withColumn("minutes_played", F.coalesce(F.col("minutes_played").cast(IntegerType()), F.lit(0)))
    .withColumn("gk_shots_on_target_against", F.coalesce(F.col("gk_shots_on_target_against").cast(IntegerType()), F.lit(0)))
    .withColumn("gk_goals_against", F.coalesce(F.col("gk_goals_against").cast(IntegerType()), F.lit(0)))
    .withColumn("gk_saves", F.coalesce(F.col("gk_saves").cast(IntegerType()), F.lit(0)))
    .withColumn("gk_save_pct", F.col("gk_save_pct").cast(DoubleType()))
)

dim_players_path = "s3://football-data-raw-01/silver/dim_players"
dim_players = spark.read.format("delta").load(dim_players_path)

fact_player_match_performance = (
    fact_player_match_stats_summary.alias("f")
    .join(
        dim_players.alias("d"),
        F.expr("f.player_id = d.player_id"),
        "inner"
    )
    .select(
        F.col("f.*"),
        F.col("d.player_name")
    )
)


fact_goal_keeper_match_performance = (
    fact_player_match_keepers_summary.alias("f")
    .join(
        dim_players.alias("d"),
        F.expr("f.player_id = d.player_id"),
        "inner"
    )
    .select(
        F.col("f.*"),
        F.col("d.player_name")
    )
)

fact_player_match_performance_path = "s3://football-data-raw-01/gold/fact_player_match_performance"
fact_player_match_performance.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(fact_player_match_performance_path)

fact_goal_keeper_match_performance_path = "s3://football-data-raw-01/gold/fact_goal_keeper_match_performance"
fact_goal_keeper_match_performance.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(fact_goal_keeper_match_performance_path)

Live - Flattening

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
import pandas as pd
import json, math

S3_BUCKET = "football-data-raw-01"
BRONZE_PATH = f"s3://{S3_BUCKET}/live/date=*/*.jsonl"
DELTA_ROOT = f"s3://{S3_BUCKET}/live/live_delta"

RAW_SCHEMA = "kafka_partition LONG, kafka_offset LONG, kafka_timestamp_ms LONG, key STRING, value STRING"

TEAM_STATS = {
    "ballPossession": "possession", "expectedGoals": "xg", "bigChanceCreated": "big_chances",
    "totalShotsOnGoal": "shots_total", "shotsOffGoal": "shots_off_target", "goalkeeperSaves": "saves",
    "passes": "passes", "accuratePasses": "passes_accurate", "fouls": "fouls", "totalTackle": "tackles",
    "interceptionWon": "interceptions", "totalClearance": "clearances", "ballRecovery": "recoveries",
    "touchesInOppBox": "touches_opp_box", "finalThirdEntries": "final_third_entries", "duelWonPercent": "duels_won_pct",
}
PLAYER_STATS = {
    "minutesPlayed": "minutes", "touches": "touches", "totalPass": "passes", "accuratePass": "passes_accurate",
    "totalShots": "shots", "shotOffTarget": "shots_off_target", "expectedGoals": "xg", "expectedAssists": "xa",
    "duelWon": "duels_won", "duelLost": "duels_lost", "fouls": "fouls", "wasFouled": "was_fouled",
    "totalClearance": "clearances", "ballRecovery": "recoveries",
}

def _r3(v):
    return round(v, 3) if isinstance(v, (int, float)) and not (isinstance(v, float) and math.isnan(v)) else v

def _jn(v):
    """jersey number -> float, or None (kept as a label, never used arithmetically)"""
    return float(v) if v is not None else None

def parse_channel(channel):
    """'match:16363651:stats' -> (16363651, 'stats'); non-match channels -> (None, None)"""
    parts = (channel or "").split(":")
    if len(parts) < 2 or parts[0] != "match" or not parts[1].isdigit():
        return None, None
    return int(parts[1]), (parts[2] if len(parts) > 2 else "match")

def flatten_team_stats(env, match_id, team_ids, offset_cols):
    p = env["data"]
    if "error" in p:                       
        return []
    ts = pd.to_datetime(env["timestamp"], unit="ms", utc=True)
    rows = []
    for block in p.get("statistics", []):
        if block["period"] != "ALL":
            continue
        for side in ("home", "away"):
            tid = team_ids.get(side)
            row = {**offset_cols, "match_id": match_id, "side": side,
                   "team_id": float(tid) if tid is not None else None,
                   "msg_type": env["type"], "event_ts": ts}
            row.update({col: None for col in TEAM_STATS.values()})
            for g in block["groups"]:
                for it in g["statisticsItems"]:
                    col = TEAM_STATS.get(it["key"])
                    if col and row[col] is None:     
                        v = it[f"{side}Value"]
                        row[col] = float(v) if v is not None else None
            rows.append(row)
    return rows

def flatten_lineups(env, match_id, team_ids, offset_cols):
    p = env["data"]
    ts = pd.to_datetime(env["timestamp"], unit="ms", utc=True)
    rows = []
    for side in ("home", "away"):
        team = p.get(side)
        if not team or not team.get("players"):        # error payload / not published yet
            continue
        ids = [x["teamId"] for x in team["players"] if x.get("teamId") is not None]
        side_team_id = max(set(ids), key=ids.count) if ids else None   # mode: an academy sub can carry a different teamId
        if side_team_id is not None:
            team_ids[side] = side_team_id                # feeds stats/incidents cells processed after this one

        for x in team["players"]:
            row = {
                **offset_cols, "match_id": match_id, "side": side,
                "team_id": float(side_team_id) if side_team_id is not None else None,
                "shirt_number": x.get("shirtNumber"), "player_name": x.get("player", {}).get("name"),
                "position": x.get("position"), "is_sub": x.get("substitute"), "is_captain": x.get("captain", False),
                "formation": team.get("formation"), "confirmed": p.get("confirmed"),
                "msg_type": env["type"], "event_ts": ts,
            }
            s = x.get("statistics", {})
            for k, col in PLAYER_STATS.items():
                v = s.get(k)
                v = float(v) if v is not None else None
                row[col] = _r3(v) if col in ("xg", "xa") else v
            rows.append(row)
    return rows

def flatten_incidents(env, match_id, team_ids, offset_cols):
    """One incidents message can hold a mix of types -> returns {table_name: [rows]}"""
    ts = pd.to_datetime(env["timestamp"], unit="ms", utc=True)
    out = {"live_goals": [], "live_cards": [], "live_subs": [], "live_var_decisions": [], "live_injury_time": []}

    for inc in env["data"].get("incidents", []):
        itype = inc.get("incidentType")
        base = {**offset_cols, "match_id": match_id, "time": inc.get("time"), "msg_type": env["type"], "event_ts": ts}

        if itype == "injuryTime":
            out["live_injury_time"].append({**base, "length": inc.get("length")})
            continue

        if itype not in ("goal", "card", "substitution", "varDecision") or inc.get("id") is None:
            continue                       

        side = "home" if inc.get("isHome") else "away"
        tid = team_ids.get(side)
        row = {**base, "id": inc["id"], "side": side, "team_id": float(tid) if tid is not None else None}

        if itype == "goal":
            row.update(home_score=inc.get("homeScore"), away_score=inc.get("awayScore"),
                       scorer_name=inc.get("player", {}).get("name"),
                       scorer_short_name=inc.get("player", {}).get("shortName"),
                       scorer_position=inc.get("player", {}).get("position"),
                       scorer_jersey_number=_jn(inc.get("player", {}).get("jerseyNumber")),
                       assist_name=inc.get("assist1", {}).get("name"),
                       assist_short_name=inc.get("assist1", {}).get("shortName"),
                       assist_position=inc.get("assist1", {}).get("position"),
                       assist_jersey_number=_jn(inc.get("assist1", {}).get("jerseyNumber")))
            out["live_goals"].append(row)
        elif itype == "card":
            row.update(incident_class=inc.get("incidentClass"), player_name=inc.get("playerName"),
                       reason=inc.get("reason"))
            out["live_cards"].append(row)
        elif itype == "substitution":
            row.update(player_in_name=inc.get("playerIn", {}).get("name"),
                       player_in_short_name=inc.get("playerIn", {}).get("shortName"),
                       player_in_position=inc.get("playerIn", {}).get("position"),
                       player_in_jersey_number=_jn(inc.get("playerIn", {}).get("jerseyNumber")),
                       player_out_name=inc.get("playerOut", {}).get("name"),
                       player_out_short_name=inc.get("playerOut", {}).get("shortName"),
                       player_out_position=inc.get("playerOut", {}).get("position"),
                       player_out_jersey_number=_jn(inc.get("playerOut", {}).get("jerseyNumber")))
            out["live_subs"].append(row)
        elif itype == "varDecision":
            row.update(incident_class=inc.get("incidentClass"), confirmed=inc.get("confirmed"))
            out["live_var_decisions"].append(row)

    return out

Live - Lineups

In [0]:
df_bronze = spark.read.schema(RAW_SCHEMA).json(BRONZE_PATH)
pdf_bronze = df_bronze.toPandas()
team_ids_by_match = {}
rows = []
for _, rec in pdf_bronze.iterrows():
    env = json.loads(rec["value"])
    match_id, kind = parse_channel(env.get("channel"))
    if match_id is None or env.get("type") not in ("snapshot", "update") or kind != "lineups":
        continue
    team_ids = team_ids_by_match.setdefault(match_id, {})
    offset_cols = {"kafka_partition": int(rec["kafka_partition"]), "kafka_offset": int(rec["kafka_offset"])}
    rows += flatten_lineups(env, match_id, team_ids, offset_cols)

live_lineups_path = f"{DELTA_ROOT}/live_lineups"

if rows:
    candidate = spark.createDataFrame(pd.DataFrame(rows))
    if DeltaTable.isDeltaTable(spark, live_lineups_path):
        existing = spark.read.format("delta").load(live_lineups_path).select("kafka_partition", "kafka_offset", "shirt_number")
        new_rows = candidate.join(existing, on=["kafka_partition", "kafka_offset", "shirt_number"], how="left_anti")
    else:
        new_rows = candidate
    if new_rows.count() > 0:
        new_rows.write.format("delta").mode("append").option("mergeSchema", "true").save(live_lineups_path)
    print(f"live_lineups: {candidate.count()} candidate rows -> {new_rows.count()} new")


Live - Stats

In [0]:

def load_team_ids():
    if not DeltaTable.isDeltaTable(spark, live_lineups_path):
        return {}
    df = spark.read.format("delta").load(live_lineups_path).select("match_id", "side", "team_id").distinct().toPandas()
    return {int(mid): dict(zip(g["side"], g["team_id"])) for mid, g in df.groupby("match_id")}

team_ids_by_match = load_team_ids()
rows = []
for _, rec in pdf_bronze.iterrows():
    env = json.loads(rec["value"])
    match_id, kind = parse_channel(env.get("channel"))
    if match_id is None or env.get("type") not in ("snapshot", "update") or kind != "stats":
        continue
    team_ids = team_ids_by_match.setdefault(match_id, {})
    offset_cols = {"kafka_partition": int(rec["kafka_partition"]), "kafka_offset": int(rec["kafka_offset"])}
    rows += flatten_team_stats(env, match_id, team_ids, offset_cols)

live_team_stats_path = f"{DELTA_ROOT}/live_team_stats"

if rows:
    candidate = spark.createDataFrame(pd.DataFrame(rows))
    if DeltaTable.isDeltaTable(spark, live_team_stats_path):
        existing = spark.read.format("delta").load(live_team_stats_path).select("kafka_partition", "kafka_offset", "side")
        new_rows = candidate.join(existing, on=["kafka_partition", "kafka_offset", "side"], how="left_anti")
    else:
        new_rows = candidate
    if new_rows.count() > 0:
        new_rows.write.format("delta").mode("append").option("mergeSchema", "true").save(live_team_stats_path)
    print(f"live_team_stats: {candidate.count()} candidate rows -> {new_rows.count()} new")

Live - Incidents

In [0]:
team_ids_by_match = load_team_ids()
incident_rows = {"live_goals": [], "live_cards": [], "live_subs": [], "live_var_decisions": [], "live_injury_time": []}

for _, rec in pdf_bronze.iterrows():
    env = json.loads(rec["value"])
    match_id, kind = parse_channel(env.get("channel"))
    if match_id is None or env.get("type") not in ("snapshot", "update") or kind != "incidents":
        continue
    team_ids = team_ids_by_match.setdefault(match_id, {})
    offset_cols = {"kafka_partition": int(rec["kafka_partition"]), "kafka_offset": int(rec["kafka_offset"])}
    per_table = flatten_incidents(env, match_id, team_ids, offset_cols)
    for t, r in per_table.items():
        incident_rows[t] += r

print({t: len(r) for t, r in incident_rows.items()})

Live - Incidents - Goal

In [0]:
live_goals_path = f"{DELTA_ROOT}/live_goals"
rows = incident_rows["live_goals"]

if rows:
    candidate = spark.createDataFrame(pd.DataFrame(rows))
    if DeltaTable.isDeltaTable(spark, live_goals_path):
        existing = spark.read.format("delta").load(live_goals_path).select("kafka_partition", "kafka_offset", "id")
        new_rows = candidate.join(existing, on=["kafka_partition", "kafka_offset", "id"], how="left_anti")
    else:
        new_rows = candidate
    if new_rows.count() > 0:
        new_rows.write.format("delta").mode("append").option("mergeSchema", "true").save(live_goals_path)
    print(f"live_goals: {candidate.count()} candidate rows -> {new_rows.count()} new")

if DeltaTable.isDeltaTable(spark, live_goals_path):
    display(spark.read.format("delta").load(live_goals_path))

Live - Incidents - Cards

In [0]:
live_cards_path = f"{DELTA_ROOT}/live_cards"
rows = incident_rows["live_cards"]

if rows:
    candidate = spark.createDataFrame(pd.DataFrame(rows))
    if DeltaTable.isDeltaTable(spark, live_cards_path):
        existing = spark.read.format("delta").load(live_cards_path).select("kafka_partition", "kafka_offset", "id")
        new_rows = candidate.join(existing, on=["kafka_partition", "kafka_offset", "id"], how="left_anti")
    else:
        new_rows = candidate
    if new_rows.count() > 0:
        new_rows.write.format("delta").mode("append").option("mergeSchema", "true").save(live_cards_path)
    print(f"live_cards: {candidate.count()} candidate rows -> {new_rows.count()} new")

if DeltaTable.isDeltaTable(spark, live_cards_path):
    display(spark.read.format("delta").load(live_cards_path))

Live - Incidents - Var

In [0]:
live_var_decisions_path = f"{DELTA_ROOT}/live_var_decisions"
rows = incident_rows["live_var_decisions"]

if rows:
    candidate = spark.createDataFrame(pd.DataFrame(rows))
    if DeltaTable.isDeltaTable(spark, live_var_decisions_path):
        existing = spark.read.format("delta").load(live_var_decisions_path).select("kafka_partition", "kafka_offset", "id")
        new_rows = candidate.join(existing, on=["kafka_partition", "kafka_offset", "id"], how="left_anti")
    else:
        new_rows = candidate
    if new_rows.count() > 0:
        new_rows.write.format("delta").mode("append").option("mergeSchema", "true").save(live_var_decisions_path)
    print(f"live_var_decisions: {candidate.count()} candidate rows -> {new_rows.count()} new")

if DeltaTable.isDeltaTable(spark, live_var_decisions_path):
    display(spark.read.format("delta").load(live_var_decisions_path))

Live - incidents - subs

In [0]:
live_subs_path = f"{DELTA_ROOT}/live_subs"
rows = incident_rows["live_subs"]

if rows:
    candidate = spark.createDataFrame(pd.DataFrame(rows))
    if DeltaTable.isDeltaTable(spark, live_subs_path):
        existing = spark.read.format("delta").load(live_subs_path).select("kafka_partition", "kafka_offset", "id")
        new_rows = candidate.join(existing, on=["kafka_partition", "kafka_offset", "id"], how="left_anti")
    else:
        new_rows = candidate
    if new_rows.count() > 0:
        new_rows.write.format("delta").mode("append").option("mergeSchema", "true").save(live_subs_path)
    print(f"live_subs: {candidate.count()} candidate rows -> {new_rows.count()} new")

if DeltaTable.isDeltaTable(spark, live_subs_path):
    display(spark.read.format("delta").load(live_subs_path))

Live - Injury Time

In [0]:
live_injury_time_path = f"{DELTA_ROOT}/live_injury_time"
rows = incident_rows["live_injury_time"]

if rows:
    candidate = spark.createDataFrame(pd.DataFrame(rows))
    if DeltaTable.isDeltaTable(spark, live_injury_time_path):
        existing = spark.read.format("delta").load(live_injury_time_path).select("kafka_partition", "kafka_offset", "time")
        new_rows = candidate.join(existing, on=["kafka_partition", "kafka_offset", "time"], how="left_anti")
    else:
        new_rows = candidate
    if new_rows.count() > 0:
        new_rows.write.format("delta").mode("append").option("mergeSchema", "true").save(live_injury_time_path)
    print(f"live_injury_time: {candidate.count()} candidate rows -> {new_rows.count()} new")

if DeltaTable.isDeltaTable(spark, live_injury_time_path):
    display(spark.read.format("delta").load(live_injury_time_path))

In [0]:
S3_BUCKET = "football-data-raw-01"
BRONZE_PATH = f"s3://{S3_BUCKET}/live/date=*/*.jsonl"
DELTA_ROOT = f"s3://{S3_BUCKET}/live/live_delta"
display(spark.read.format("delta").load(f"{DELTA_ROOT}/live_lineups").printSchema())